# Conformer-ByteHum: Evaluation, Retrieval & Search
Notebook doc lap de nap checkpoint fine-tuned, danh gia retrieval, DTW, FAISS va truy van humming.


### 1.1 Cai dat va nap cac thu vien can thiet
Nap cac thu vien chuan cho xu ly am thanh (librosa, soundfile), tinh toan tensor (PyTorch) va tim kiem vector (FAISS). Tu dong cai dat bo sung neu chay tren Kaggle/Colab.

In [ ]:
from tqdm import tqdm
import os
import sys
import subprocess
import time
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display as ipd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from typing import Dict, List, Optional, Tuple

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules

if IS_KAGGLE or IS_COLAB:
    print("[*] Phat hien moi truong dam may (Kaggle / Colab). Kiem tra thu vien...")
    try:
        import faiss
    except ImportError:
        print("[!] Dang cai dat faiss-cpu tu dong...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu', 'soundfile', 'librosa', 'typeguard<3'])
        import faiss
else:
    import faiss

try:
    import soundfile as sf
    import librosa
    import librosa.display
    import typeguard
    if not hasattr(typeguard, 'check_argument_types'):
        raise ImportError('CoverHunter requires typeguard<3')
except ImportError:
    if not (IS_KAGGLE or IS_COLAB):
        raise
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'librosa', 'typeguard<3'])
    sys.modules.pop('typeguard', None)
    import typeguard
    import soundfile as sf
    import librosa
    import librosa.display


### 1.2 Co dinh Seed va kiem tra phan cung GPU
Thiet lap seed ngau nhien (seed=42) de dam bao tinh tai lap cua ket qua thuc nghiem, dong thoi kiem tra thong so card do hoa (GPU VRAM).

In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Thiet bi tinh toan: {device}")
if device.type == 'cuda':
    print(f"[+] GPU Model        : {torch.cuda.get_device_name(0)}")
    print(f"[+] GPU VRAM         : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### 1.3 Dinh nghia ham dinh vi duong dan Dataset va Checkpoint
Tu dong giai nen datasets.zip tren Kaggle neu chua giai nen, sau do dinh vi thang vao cac thu muc con: dataset_split, 2_music_database, 4_vocal_separated va checkpoint Stage 1.

In [ ]:
# Tu dong giai nen dataset (.zip hoac .bin) tren Kaggle neu chua giai nen
if IS_KAGGLE and not os.path.exists("/kaggle/working/datasets") and not os.path.exists("/kaggle/input/dataset-qbh/datasets"):
    zip_file = None
    if os.path.exists("/kaggle/input"):
        for r, _, files in os.walk("/kaggle/input"):
            for f in files:
                if any(f.lower().startswith("dataset") and f.lower().endswith(ext) for ext in [".zip", ".bin", ".dat"]):
                    zip_file = os.path.join(r, f)
                    break
                elif f.endswith(".zip"):
                    zip_file = os.path.join(r, f)
                    break
            if zip_file:
                break
                
    if zip_file:
        print(f"[*] Phat hien {zip_file}, dang giai nen vao /kaggle/working...")
        subprocess.run(['unzip', '-q', zip_file, '-d', '/kaggle/working'], check=True)
        print("[+] Giai nen dataset thanh cong!")

def find_dataset_and_checkpoint():
    """Dinh vi tu dong dataset, music database, vocal stems va checkpoint Stage 1."""
    base_dir = "/kaggle/working" if IS_KAGGLE else (os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath("."))
    
    # 1. Dinh vi thu muc dataset (uu tien dung cau truc Kaggle Input: /kaggle/input/dataset-qbh/datasets)
    search_roots = [
        "/kaggle/input/dataset-qbh/datasets",
        "/kaggle/input/dataset-qbh",
        "/kaggle/working/datasets",
        "/kaggle/working",
        os.path.join(base_dir, "02_stage2_humming_qbh", "datasets"),
        os.path.join(base_dir, "datasets"),
        base_dir
    ]
    if IS_KAGGLE and os.path.exists("/kaggle/input"):
        search_roots += [os.path.join("/kaggle/input", d, "datasets") for d in os.listdir("/kaggle/input") if os.path.isdir(os.path.join("/kaggle/input", d))]
        search_roots += [os.path.join("/kaggle/input", d) for d in os.listdir("/kaggle/input") if os.path.isdir(os.path.join("/kaggle/input", d))]
        
    dataset_split_dir = None
    music_db_dir = None
    vocal_dir = None
    
    for r in search_roots:
        if not r or not os.path.exists(r):
            continue
        if not dataset_split_dir:
            if os.path.exists(os.path.join(r, "dataset_split")):
                dataset_split_dir = os.path.join(r, "dataset_split")
            elif os.path.exists(os.path.join(r, "train")) and os.path.exists(os.path.join(r, "test")):
                dataset_split_dir = r
        if not music_db_dir and os.path.exists(os.path.join(r, "2_music_database")):
            music_db_dir = os.path.join(r, "2_music_database")
        if not vocal_dir and os.path.exists(os.path.join(r, "4_vocal_separated")):
            vocal_dir = os.path.join(r, "4_vocal_separated")
            
    # Quet de quy neu chua tim thay du
    if not dataset_split_dir or not music_db_dir:
        for root, dirs, _ in os.walk("/kaggle" if IS_KAGGLE else base_dir):
            if not dataset_split_dir and "dataset_split" in dirs:
                dataset_split_dir = os.path.join(root, "dataset_split")
            if not music_db_dir and "2_music_database" in dirs:
                music_db_dir = os.path.join(root, "2_music_database")
            if not vocal_dir and "4_vocal_separated" in dirs:
                vocal_dir = os.path.join(root, "4_vocal_separated")
            if dataset_split_dir and music_db_dir and vocal_dir:
                break
                
    if not dataset_split_dir: dataset_split_dir = os.path.join(base_dir, "02_stage2_humming_qbh", "datasets", "dataset_split")
    if not music_db_dir: music_db_dir = os.path.join(base_dir, "02_stage2_humming_qbh", "datasets", "2_music_database")
    # Neu khong co thu muc vocal rieng thi fallback doc truc tiep tu music_db
    if not vocal_dir: vocal_dir = music_db_dir

    # 2. Dinh vi checkpoint pre-trained Stage 1 (coverhunter_shs100k_pretrained.pth)
    ckpt_path = None
    if IS_KAGGLE:
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(".pth") and any(k in f.lower() for k in ["coverhunter", "shs100k"]):
                    ckpt_path = os.path.join(root, f)
                    break
            if ckpt_path: break
    else:
        for c in [
            os.path.join(base_dir, "checkpoints", "coverhunter_shs100k_pretrained.pth"),
            os.path.join(base_dir, "02_stage2_humming_qbh", "checkpoints", "coverhunter_shs100k_pretrained.pth"),
            os.path.abspath("checkpoints/coverhunter_shs100k_pretrained.pth")
        ]:
            if os.path.exists(c):
                ckpt_path = c
                break

    return dataset_split_dir, music_db_dir, vocal_dir, ckpt_path


### 1.4 Thiet lap thu muc luu tru ket qua va kiem tra duong dan du lieu
Thuc thi do tim duong dan, tao thu muc luu tru checkpoint (`checkpoints`) va kiem tra su ton tai cua cac thu muc du lieu.

In [ ]:
DATASET_SPLIT_DIR, MUSIC_DB_DIR, VOCAL_DIR, STAGE1_CKPT = find_dataset_and_checkpoint()

# Cau hinh thu muc xuat ra (Output Working Directory)
if IS_KAGGLE:
    OUTPUT_DIR = "/kaggle/working"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
else:
    OUTPUT_DIR = os.path.abspath(".")
    base_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
    if os.path.exists(os.path.join(base_dir, "checkpoints")):
        CHECKPOINT_DIR = os.path.join(base_dir, "checkpoints")
    elif os.path.exists(os.path.join(base_dir, "02_stage2_humming_qbh", "checkpoints")):
        CHECKPOINT_DIR = os.path.join(base_dir, "02_stage2_humming_qbh", "checkpoints")
    else:
        CHECKPOINT_DIR = os.path.abspath("checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

TRAIN_DIR = os.path.join(DATASET_SPLIT_DIR, "train") if DATASET_SPLIT_DIR and os.path.exists(DATASET_SPLIT_DIR) else None
VAL_DIR = os.path.join(DATASET_SPLIT_DIR, "val") if DATASET_SPLIT_DIR and os.path.exists(DATASET_SPLIT_DIR) else None
TEST_DIR = os.path.join(DATASET_SPLIT_DIR, "test") if DATASET_SPLIT_DIR and os.path.exists(DATASET_SPLIT_DIR) else None

print("[+] KET QUA DINH VI DUONG DAN TU DONG:")
print(f"    - Moi truong thuc thi            : {'KAGGLE' if IS_KAGGLE else ('COLAB' if IS_COLAB else 'LOCAL')}")
print(f"    - Thu muc Train Humming          : {TRAIN_DIR} (Ton tai: {os.path.exists(str(TRAIN_DIR)) if TRAIN_DIR else False})")
print(f"    - Thu muc Val Humming            : {VAL_DIR} (Ton tai: {os.path.exists(str(VAL_DIR)) if VAL_DIR else False})")
print(f"    - Thu muc Test Humming           : {TEST_DIR} (Ton tai: {os.path.exists(str(TEST_DIR)) if TEST_DIR else False})")
print(f"    - Kho Nhac Goc 300 Bai (MP3)     : {MUSIC_DB_DIR} (Ton tai: {os.path.exists(str(MUSIC_DB_DIR)) if MUSIC_DB_DIR else False})")
print(f"    - Checkpoint Stage 1 Pre-trained : {STAGE1_CKPT} (Ton tai: {os.path.exists(str(STAGE1_CKPT)) if STAGE1_CKPT else False})")
print("    - CoverHunter Backbone           : Da nhung truc tiep trong notebook")
print(f"    - Thu muc Checkpoint             : {CHECKPOINT_DIR} (Ton tai: {os.path.exists(CHECKPOINT_DIR)})")
print(f"    - Thu muc luu ket qua (Output)   : {OUTPUT_DIR}")


In [ ]:
# CoverHunter backbone is embedded to keep this notebook self-contained.
import logging
import math
from typing import Union
from typeguard import check_argument_types

class BasicModel(torch.nn.Module):
  """Minimal CoverHunter base class required by the embedded inference model."""

  def model_size(self) -> int:
    return sum(parameter.numel() for parameter in self.parameters())



class Linear(torch.nn.Module):
  def __init__(self, in_dim, out_dim, bias=True, w_init_gain='linear'):
    super(Linear, self).__init__()
    self.linear_layer = torch.nn.Linear(in_dim, out_dim, bias=bias)

    torch.nn.init.xavier_uniform_(
      self.linear_layer.weight,
      gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, x):
    return self.linear_layer(x)


class Conv1d(torch.nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=1, stride=1,
               padding=None, dilation=1, bias=True, w_init_gain='linear'):
    super(Conv1d, self).__init__()
    if padding is None:
      assert (kernel_size % 2 == 1)
      padding = int(dilation * (kernel_size - 1) / 2)

    self.conv = torch.nn.Conv1d(in_channels, out_channels,
                                kernel_size=kernel_size, stride=stride,
                                padding=padding, dilation=dilation,
                                bias=bias)

    torch.nn.init.xavier_uniform_(
      self.conv.weight, gain=torch.nn.init.calculate_gain(w_init_gain))
    return

  def forward(self, signal):
    conv_signal = self.conv(signal)
    return conv_signal


class CenterLoss(nn.Module):
  """Center loss.

  Reference:
    A Discriminative Feature Learning Approach for Deep Face Recognition.

  Args:
      num_classes (int): number of classes.
      feat_dim (int): feature dimension.
  """

  def __init__(self, num_classes=10, feat_dim=2, use_gpu=True):
    super(CenterLoss, self).__init__()
    self.num_classes = num_classes
    self.feat_dim = feat_dim
    self.use_gpu = use_gpu
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    if self.use_gpu:
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).cuda())
      self.centers = nn.Parameter(
        torch.randn(self.num_classes, self.feat_dim).to("cuda"))
    else:
      self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim))
    logging.info("CenterLoss: num_classes({}), feat_dim({})".format(
      num_classes, feat_dim))
    return

  def forward(self, x, labels):
    """
    Args:
        x: feature matrix with shape (batch_size, feat_dim).
        labels: ground truth labels with shape (batch_size).
    """
    batch_size = x.size(0)
    distmat = torch.pow(x, 2).sum(dim=1, keepdim=True). \
                expand(batch_size, self.num_classes) + \
              torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(
                self.num_classes, batch_size).t()
    distmat.addmm_(1, -2, x, self.centers.t())

    classes = torch.arange(self.num_classes).long()
    if self.use_gpu:
      classes = classes.cuda()
    labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
    mask = labels.eq(classes.expand(batch_size, self.num_classes))

    dist = distmat * mask.float()
    loss = dist.clamp(min=1e-12, max=1e+12).sum() / batch_size

    return loss


class FocalLoss(nn.Module):
  """ Focal Loss implement for https://arxiv.org/abs/1708.02002 """

  def __init__(self, gamma: float = 2., alpha: Optional[List] = None, num_cls: int = -1,
               reduction: str = 'mean'):

    super(FocalLoss, self).__init__()
    assert check_argument_types()
    if reduction not in ['mean', 'sum']:
      raise NotImplementedError(
        'Reduction {} not implemented.'.format(reduction))
    self._reduction = reduction
    self._alpha = alpha
    self._gamma = gamma
    if alpha is not None:
      assert len(alpha) <= num_cls, "{} != {}".format(len(alpha), num_cls)
      self._alpha = torch.tensor(self._alpha)
    self._eps = np.finfo(float).eps
    logging.info("Init Focal loss with gamma:{}".format(gamma))
    return

  def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """ compute focal loss for pred and label

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss
    """
    b = y_pred.size(0)
    y_pred_softmax = torch.nn.Softmax(dim=1)(y_pred) + self._eps
    ce = -torch.log(y_pred_softmax)
    ce = ce.gather(1, y_true.view(-1, 1))

    y_pred_softmax = y_pred_softmax.gather(1, y_true.view(-1, 1))
    weight = torch.pow(torch.sub(1., y_pred_softmax), self._gamma)

    if self._alpha is not None:
      self._alpha = self._alpha.to(y_pred.device)
      alpha = self._alpha.gather(0, y_true.view(-1))
      alpha = alpha.unsqueeze(1)
      alpha = alpha / torch.sum(alpha) * b
      weight = torch.mul(alpha, weight)
    fl_loss = torch.mul(weight, ce).squeeze(1)
    return self._reduce(fl_loss)

  def forward_with_onehot(self, y_pred: torch.Tensor,
                          y_true: torch.Tensor) -> torch.Tensor:
    """ Another implement for "forward" with onehot label matrix.

    It is not good because when ce_embed size is large, more memory will used.

    Args:
      y_pred: [batch_size, num_cls]
      y_true: [batch_size]

    Returns:
      loss

    """
    y_pred = torch.nn.Softmax(dim=1)(y_pred)
    y_true = torch.nn.functional.one_hot(y_true, self.num_cls)

    eps = np.finfo(float).eps
    y_pred = y_pred + eps
    ce = torch.mul(y_true, -torch.log(y_pred))
    weight = torch.mul(y_true, torch.pow(torch.sub(1., y_pred), self._gamma))
    fl_loss = torch.mul(self.alpha, torch.mul(weight, ce))

    fl_loss, _ = torch.max(fl_loss, dim=1)
    return self._reduce(fl_loss)

  def _reduce(self, x):
    if self._reduction == 'mean':
      return torch.mean(x)
    else:
      return torch.sum(x)


class HardTripletLoss(nn.Module):
  """Hard/Hardest Triplet Loss
    (pytorch implementation of https://omoindrot.github.io/triplet-loss)
    For each anchor, we get the hardest positive and hardest negative to form a triplet.
    """

  def __init__(self, margin=0.1):
    """ Args:
      margin: margin for triplet loss
    """
    super(HardTripletLoss, self).__init__()
    self._margin = margin
    return

  def forward(self, embeddings, labels):
    """
    Args:
        labels: labels of the batch, of size (batch_size,)
        embeddings: tensor of shape (batch_size, embed_dim)

    Returns:
        triplet_loss: scalar tensor containing the triplet loss
    """
    pairwise_dist = self._pairwise_distance(embeddings, squared=False)

    mask_anchor_positive = self._get_anchor_positive_triplet_mask(
      labels).float()
    valid_positive_dist = pairwise_dist * mask_anchor_positive
    hardest_positive_dist, _ = torch.max(valid_positive_dist, dim=1,
                                         keepdim=True)

    # Get the hardest negative pairs
    mask_negative = self._get_anchor_negative_triplet_mask(labels).float()
    max_negative_dist, _ = torch.max(pairwise_dist, dim=1, keepdim=True)
    negative_dist = pairwise_dist + max_negative_dist * (1.0 - mask_negative)
    hardest_negative_dist, _ = torch.min(negative_dist, dim=1, keepdim=True)

    # Combine biggest d(a, p) and smallest d(a, n) into final triplet loss
    triplet_loss = F.relu(
      hardest_positive_dist - hardest_negative_dist + self._margin)
    triplet_loss = torch.mean(triplet_loss)
    return triplet_loss

  @staticmethod
  def _pairwise_distance(x, squared=False, eps=1e-16):
    # Compute the 2D matrix of distances between all the embeddings.

    cor_mat = torch.matmul(x, x.t())
    norm_mat = cor_mat.diag()
    distances = norm_mat.unsqueeze(1) - 2 * cor_mat + norm_mat.unsqueeze(0)
    distances = F.relu(distances)

    if not squared:
      mask = torch.eq(distances, 0.0).float()
      distances = distances + mask * eps
      distances = torch.sqrt(distances)
      distances = distances * (1.0 - mask)
    return distances

  @staticmethod
  def _get_anchor_positive_triplet_mask(labels):
    """Return a 2D mask where mask[a, p] is True, if a and p are distinct and
       have same label.

    """
    # device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    device = labels.device
    indices_not_equal = torch.eye(labels.shape[0]).to(device).byte() ^ 1
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = indices_not_equal * labels_equal
    return mask

  @staticmethod
  def _get_anchor_negative_triplet_mask(labels):
    """Return a 2D mask where mask[a, n] is True iff a and n have distinct labels.
    """
    labels_equal = torch.unsqueeze(labels, 0) == torch.unsqueeze(labels, 1)
    mask = labels_equal ^ 1
    return mask


def subsequent_chunk_mask(
    size: int,
    chunk_size: int,
    num_left_chunks: int = -1,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
  """Create mask for subsequent steps (size, size) with chunk size,
       this is for streaming encoder

    Args:
        size (int): size of mask
        chunk_size (int): size of chunk
        num_left_chunks (int): number of left chunks
            <0: use full chunk
            >=0: use num_left_chunks
        device (torch.device): "cpu" or "cuda" or torch.Tensor.device

    Returns:
        torch.Tensor: mask

    Examples:
        >>> subsequent_chunk_mask(4, 2)
        [[1, 1, 0, 0],
         [1, 1, 0, 0],
         [1, 1, 1, 1],
         [1, 1, 1, 1]]
    """
  ret = torch.zeros(size, size, device=device, dtype=torch.bool)
  for i in range(size):
    if num_left_chunks < 0:
      start = 0
    else:
      start = max((i // chunk_size - num_left_chunks) * chunk_size, 0)
    ending = min((i // chunk_size + 1) * chunk_size, size)
    ret[i, start:ending] = True
  return ret


def add_optional_chunk_mask(xs: torch.Tensor, masks: torch.Tensor,
                            use_dynamic_chunk: bool,
                            use_dynamic_left_chunk: bool,
                            decoding_chunk_size: int, static_chunk_size: int,
                            num_decoding_left_chunks: int):
  """ Apply optional mask for encoder.

    Args:
        masks:
        xs (torch.Tensor): padded input, (B, L, D), L for max length
        mask (torch.Tensor): mask for xs, (B, 1, L)
        use_dynamic_chunk (bool): whether to use dynamic chunk or not
        use_dynamic_left_chunk (bool): whether to use dynamic left chunk for
            training.
        decoding_chunk_size (int): decoding chunk size for dynamic chunk, it's
            0: default for training, use random dynamic chunk.
            <0: for decoding, use full chunk.
            >0: for decoding, use fixed chunk size as set.
        static_chunk_size (int): chunk size for static chunk training/decoding
            if it's greater than 0, if use_dynamic_chunk is true,
            this parameter will be ignored
        num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
            >=0: use num_decoding_left_chunks
            <0: use all left chunks

    Returns:
        torch.Tensor: chunk mask of the input xs.
    """
  # Whether to use chunk mask or not
  if use_dynamic_chunk:
    max_len = xs.size(1)
    if decoding_chunk_size < 0:
      chunk_size = max_len
      num_left_chunks = -1
    elif decoding_chunk_size > 0:
      chunk_size = decoding_chunk_size
      num_left_chunks = num_decoding_left_chunks
    else:
      # chunk size is either [1, 25] or full context(max_len).
      # Since we use 4 times subsampling and allow up to 1s(100 frames)
      # delay, the maximum frame is 100 / 4 = 25.
      chunk_size = torch.randint(1, max_len, (1,)).item()
      num_left_chunks = -1
      if chunk_size > max_len // 2:
        chunk_size = max_len
      else:
        chunk_size = chunk_size % 25 + 1
        if use_dynamic_left_chunk:
          max_left_chunks = (max_len - 1) // chunk_size
          num_left_chunks = torch.randint(0, max_left_chunks,
                                          (1,)).item()
    chunk_masks = subsequent_chunk_mask(xs.size(1), chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  elif static_chunk_size > 0:
    num_left_chunks = num_decoding_left_chunks
    chunk_masks = subsequent_chunk_mask(xs.size(1), static_chunk_size,
                                        num_left_chunks,
                                        xs.device)  # (L, L)
    chunk_masks = chunk_masks.unsqueeze(0)  # (1, L, L)
    chunk_masks = masks & chunk_masks  # (B, L, L)
  else:
    chunk_masks = masks
  return chunk_masks


def make_pad_mask(lengths: torch.Tensor, max_len: int = 0) -> torch.Tensor:
  """Make mask tensor containing indices of padded part.

    See description of make_non_pad_mask.

    Args:
        lengths (torch.Tensor): Batch of lengths (B,).
        max_len:

    Returns:
        torch.Tensor: Mask tensor containing indices of padded part.

    Examples:
        >>> make_pad_mask(torch.tensor([5, 3, 2]))
        masks = [[0, 0, 0, 0 ,0],
                 [0, 0, 0, 1, 1],
                 [0, 0, 1, 1, 1]]
    """
  batch_size = lengths.size(0)
  max_len = max_len if max_len > 0 else lengths.max().item()
  seq_range = torch.arange(0,
                           max_len,
                           dtype=torch.int64,
                           device=lengths.device)
  seq_range_expand = seq_range.unsqueeze(0).expand(batch_size, max_len)
  seq_length_expand = lengths.unsqueeze(-1)
  mask = seq_range_expand >= seq_length_expand
  return mask


class Swish(torch.nn.Module):
  """Construct an Swish object."""

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """Return Swish activation function."""
    return x * torch.sigmoid(x)


def get_activation(act):
  """Return activation function."""
  activation_funcs = {
    "hardtanh": torch.nn.Hardtanh,
    "tanh": torch.nn.Tanh,
    "relu": torch.nn.ReLU,
    "selu": torch.nn.SELU,
    "swish": getattr(torch.nn, "SiLU", Swish),
    "gelu": torch.nn.GELU
  }
  return activation_funcs[act]()


class PositionalEncoding(torch.nn.Module):
  """Positional encoding.

    :param int d_model: embedding dim
    :param float dropout_rate: dropout rate
    :param int max_len: maximum input length

    PE(pos, 2i)   = sin(pos/(10000^(2i/dmodel)))
    PE(pos, 2i+1) = cos(pos/(10000^(2i/dmodel)))
    """

  def __init__(self,
               d_model: int,
               dropout_rate: float,
               max_len: int = 5000,
               reverse: bool = False):
    """Construct an PositionalEncoding object."""
    super().__init__()
    self.d_model = d_model
    self.xscale = math.sqrt(self.d_model)
    self.dropout = torch.nn.Dropout(p=dropout_rate)
    self.max_len = max_len

    self.pe = torch.zeros(self.max_len, self.d_model)
    position = torch.arange(0, self.max_len,
                            dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
      torch.arange(0, self.d_model, 2, dtype=torch.float32) *
      -(math.log(10000.0) / self.d_model))
    self.pe[:, 0::2] = torch.sin(position * div_term)
    self.pe[:, 1::2] = torch.cos(position * div_term)
    self.pe = self.pe.unsqueeze(0)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Add positional encoding.

        Args:
            x (torch.Tensor): Input. Its shape is (batch, time, ...)
            offset (int, torch.tensor): position offset

        Returns:
            torch.Tensor: Encoded tensor. Its shape is (batch, time, ...)
            torch.Tensor: for compatibility to RelPositionalEncoding
        """

    self.pe = self.pe.to(x.device)
    pos_emb = self.position_encoding(offset, x.size(1), False)
    x = x * self.xscale + pos_emb
    return self.dropout(x), self.dropout(pos_emb)

  def position_encoding(self, offset: Union[int, torch.Tensor], size: int,
                        apply_dropout: bool = True) -> torch.Tensor:
    """ For getting encoding in a streaming fashion

        Attention!!!!!
        we apply dropout only once at the whole utterance level in a none
        streaming way, but will call this function several times with
        increasing input size in a streaming scenario, so the dropout will
        be applied several times.

        Args:
            apply_dropout:
            offset (int or torch.tensor): start offset
            size (int): required size of position encoding

        Returns:
            torch.Tensor: Corresponding encoding
        """
    # How to subscript a Union type:
    #   https://github.com/pytorch/pytorch/issues/69434
    if isinstance(offset, int):
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    elif isinstance(offset, torch.Tensor) and offset.dim() == 0:  # scalar
      assert offset + size < self.max_len
      pos_emb = self.pe[:, offset:offset + size]
    else:  # for batched streaming decoding on GPU
      assert torch.max(offset) + size < self.max_len
      index = offset.unsqueeze(1) + torch.arange(0, size).to(offset.device)  # B X T
      flag = index > 0
      # remove negative offset
      index = index * flag
      pos_emb = F.embedding(index, self.pe[0])  # B X T X d_model

    if apply_dropout:
      pos_emb = self.dropout(pos_emb)
    return pos_emb


class RelPositionalEncoding(PositionalEncoding):
  """Relative positional encoding module.
    See : Appendix B in https://arxiv.org/abs/1901.02860
    Args:
        d_model (int): Embedding dimension.
        dropout_rate (float): Dropout rate.
        max_len (int): Maximum input length.
    """

  def __init__(self, d_model: int, dropout_rate: float, max_len: int = 5000):
    """Initialize class."""
    super().__init__(d_model, dropout_rate, max_len, reverse=True)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute positional encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, `*`).
        Returns:
            torch.Tensor: Encoded tensor (batch, time, `*`).
            torch.Tensor: Positional embedding tensor (1, time, `*`).
        """
    self.pe = self.pe.to(x.device)
    x = x * self.xscale
    pos_emb = self.position_encoding(offset, x.size(1), False)
    return self.dropout(x), self.dropout(pos_emb)


class NoPositionalEncoding(torch.nn.Module):
  """ No position encoding
    """

  def __init__(self, d_model: int, dropout_rate: float):
    super().__init__()
    self.d_model = d_model
    self.dropout = torch.nn.Dropout(p=dropout_rate)

  def forward(self,
              x: torch.Tensor,
              offset: Union[int, torch.Tensor] = 0) \
      -> Tuple[torch.Tensor, torch.Tensor]:
    """ Just return zero vector for interface compatibility
        """
    pos_emb = torch.zeros(1, x.size(1), self.d_model).to(x.device)
    return self.dropout(x), pos_emb

  def position_encoding(
      self, offset: Union[int, torch.Tensor], size: int) -> torch.Tensor:
    return torch.zeros(1, size, self.d_model)


class BaseSubsampling(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.right_context = 0
    self.subsampling_rate = 1

  def position_encoding(self, offset: Union[int, torch.Tensor],
                        size: int) -> torch.Tensor:
    return self.pos_enc.position_encoding(offset, size)


class LinearNoSubsampling(BaseSubsampling):
  """Linear transform the input without subsampling

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an linear object."""
    super().__init__()
    self.out = torch.nn.Sequential(
      torch.nn.Linear(idim, odim),
      torch.nn.LayerNorm(odim, eps=1e-5),
      torch.nn.Dropout(dropout_rate),
    )
    self.pos_enc = pos_enc_class
    self.right_context = 0
    self.subsampling_rate = 1

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Input x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: linear input tensor (#batch, time', odim),
                where time' = time .
            torch.Tensor: linear input mask (#batch, 1, time'),
                where time' = time .

        """
    x = self.out(x)
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask


class Conv2dSubsampling4(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/4 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling4 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.out = torch.nn.Sequential(
      torch.nn.Linear(odim * (((idim - 1) // 2 - 1) // 2), odim))
    self.pos_enc = pos_enc_class
    # The right context for every conv layer is computed by:
    # (kernel_size - 1) * frame_rate_of_this_layer
    self.subsampling_rate = 4
    # 6 = (3 - 1) * 1 + (3 - 1) * 2
    self.right_context = 6

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 4.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 4.
            torch.Tensor: positional encoding

        """
    x = x.unsqueeze(1)  # (b, c=1, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.out(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2]


class Conv2dSubsampling8(BaseSubsampling):
  """Convolutional 2D subsampling (to 1/8 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, idim: int, odim: int, dropout_rate: float,
               pos_enc_class: torch.nn.Module):
    """Construct an Conv2dSubsampling8 object."""
    super().__init__()
    self.conv = torch.nn.Sequential(
      torch.nn.Conv2d(1, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
      torch.nn.Conv2d(odim, odim, 3, 2),
      torch.nn.ReLU(),
    )
    self.linear = torch.nn.Linear(
      odim * ((((idim - 1) // 2 - 1) // 2 - 1) // 2), odim)
    self.pos_enc = pos_enc_class
    self.subsampling_rate = 8
    # 14 = (3 - 1) * 1 + (3 - 1) * 2 + (3 - 1) * 4
    self.right_context = 14

  def forward(
      self,
      x: torch.Tensor,
      x_mask: torch.Tensor,
      offset: Union[int, torch.Tensor] = 0
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 8.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 8.
            torch.Tensor: positional encoding
        """
    x = x.unsqueeze(1)  # (b, c, t, f)
    x = self.conv(x)
    b, c, t, f = x.size()
    x = self.linear(x.transpose(1, 2).contiguous().view(b, t, c * f))
    x, pos_emb = self.pos_enc(x, offset)
    return x, pos_emb, x_mask[:, :, :-2:2][:, :, :-2:2][:, :, :-2:2]


class MultiHeadedAttention(nn.Module):
  """Multi-Head Attention layer.

    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.

    """

  def __init__(self, n_head: int, n_feat: int, dropout_rate: float):
    """Construct an MultiHeadedAttention object."""
    super().__init__()
    assert n_feat % n_head == 0
    # We assume d_v always equals d_k
    self.d_k = n_feat // n_head
    self.h = n_head
    self.linear_q = nn.Linear(n_feat, n_feat)
    self.linear_k = nn.Linear(n_feat, n_feat)
    self.linear_v = nn.Linear(n_feat, n_feat)
    self.linear_out = nn.Linear(n_feat, n_feat)
    self.dropout = nn.Dropout(p=dropout_rate)

  def forward_qkv(
      self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Transform query, key and value.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).

        Returns:
            torch.Tensor: Transformed query tensor, size
                (#batch, n_head, time1, d_k).
            torch.Tensor: Transformed key tensor, size
                (#batch, n_head, time2, d_k).
            torch.Tensor: Transformed value tensor, size
                (#batch, n_head, time2, d_k).

        """
    n_batch = query.size(0)
    q = self.linear_q(query).view(n_batch, -1, self.h, self.d_k)
    k = self.linear_k(key).view(n_batch, -1, self.h, self.d_k)
    v = self.linear_v(value).view(n_batch, -1, self.h, self.d_k)
    q = q.transpose(1, 2)  # (batch, head, time1, d_k)
    k = k.transpose(1, 2)  # (batch, head, time2, d_k)
    v = v.transpose(1, 2)  # (batch, head, time2, d_k)

    return q, k, v

  def forward_attention(
      self, value: torch.Tensor, scores: torch.Tensor,
      mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool)
  ) -> torch.Tensor:
    """Compute attention context vector.

        Args:
            value (torch.Tensor): Transformed value, size
                (#batch, n_head, time2, d_k).
            scores (torch.Tensor): Attention score, size
                (#batch, n_head, time1, time2).
            mask (torch.Tensor): Mask, size (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.

        Returns:
            torch.Tensor: Transformed value (#batch, time1, d_model)
                weighted by the attention score (#batch, time1, time2).

        """
    n_batch = value.size(0)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be True?
    #   1. onnx(16/4) [WHY? Because we feed real cache & real mask for the
    #           1st chunk to ease the onnx export.]
    #   2. pytorch training
    if mask.size(2) > 0:  # time2 > 0
      mask = mask.unsqueeze(1).eq(0)  # (batch, 1, *, time2)
      # For last chunk, time2 might be larger than scores.size(-1)
      mask = mask[:, :, :, :scores.size(-1)]  # (batch, 1, *, time2)
      scores = scores.masked_fill(mask, -float('inf'))
      attn = torch.softmax(scores, dim=-1).masked_fill(
        mask, 0.0)  # (batch, head, time1, time2)
    # NOTE(xcsong): When will `if mask.size(2) > 0` be False?
    #   1. onnx(16/-1, -1/-1, 16/0)
    #   2. jit (16/-1, -1/-1, 16/0, 16/4)
    else:
      attn = torch.softmax(scores, dim=-1)  # (batch, head, time1, time2)

    p_attn = self.dropout(attn)
    x = torch.matmul(p_attn, value)  # (batch, head, time1, d_k)
    x = (x.transpose(1, 2).contiguous().view(n_batch, -1,
                                             self.h * self.d_k)
         )  # (batch, time1, d_model)

    return self.linear_out(x)  # (batch, time1, d_model)

  def forward(self, query: torch.Tensor, key: torch.Tensor,
              value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot product attention.

        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2).
                1.When applying cross attention between decoder and encoder,
                the batch padding mask for input is in (#batch, 1, T) shape.
                2.When applying self attention of encoder,
                the mask is in (#batch, T, T)  shape.
                3.When applying self attention of decoder,
                the mask is in (#batch, L, L)  shape.
                4.If the different position in decoder see different block
                of the encoder, such as Mocha, the passed in mask could be
                in (#batch, L, T) shape. But there is no such case in current
                Wenet.
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`


        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`

        """
    q, k, v = self.forward_qkv(query, key, value)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
    return self.forward_attention(v, scores, mask), new_cache


class RelPositionMultiHeadedAttention(MultiHeadedAttention):
  """Multi-Head Attention layer with relative position encoding.
    Paper: https://arxiv.org/abs/1901.02860
    Args:
        n_head (int): The number of heads.
        n_feat (int): The number of features.
        dropout_rate (float): Dropout rate.
    """

  def __init__(self, n_head, n_feat, dropout_rate):
    """Construct an RelPositionMultiHeadedAttention object."""
    super().__init__(n_head, n_feat, dropout_rate)
    # linear transformation for positional encoding
    self.linear_pos = nn.Linear(n_feat, n_feat, bias=False)
    # these two learnable bias are used in matrix c and matrix d
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    self.pos_bias_u = nn.Parameter(torch.Tensor(self.h, self.d_k))
    self.pos_bias_v = nn.Parameter(torch.Tensor(self.h, self.d_k))
    torch.nn.init.xavier_uniform_(self.pos_bias_u)
    torch.nn.init.xavier_uniform_(self.pos_bias_v)

  def rel_shift(self, x, zero_triu: bool = False):
    """Compute relative positinal encoding.
        Args:
            x (torch.Tensor): Input tensor (batch, time, size).
            zero_triu (bool): If true, return the lower triangular part of
                the matrix.
        Returns:
            torch.Tensor: Output tensor.
        """

    zero_pad = torch.zeros((x.size()[0], x.size()[1], x.size()[2], 1),
                           device=x.device,
                           dtype=x.dtype)
    x_padded = torch.cat([zero_pad, x], dim=-1)

    x_padded = x_padded.view(x.size()[0],
                             x.size()[1],
                             x.size(3) + 1, x.size(2))
    x = x_padded[:, :, 1:].view_as(x)

    if zero_triu:
      ones = torch.ones((x.size(2), x.size(3)))
      x = x * torch.tril(ones, x.size(3) - x.size(2))[None, None, :, :]

    return x

  def forward(self, query: torch.Tensor,
              key: torch.Tensor, value: torch.Tensor,
              mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
              pos_emb: torch.Tensor = torch.empty(0),
              cache: torch.Tensor = torch.zeros((0, 0, 0, 0))
              ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute 'Scaled Dot Product Attention' with rel. positional encoding.
        Args:
            query (torch.Tensor): Query tensor (#batch, time1, size).
            key (torch.Tensor): Key tensor (#batch, time2, size).
            value (torch.Tensor): Value tensor (#batch, time2, size).
            mask (torch.Tensor): Mask tensor (#batch, 1, time2) or
                (#batch, time1, time2), (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): Positional embedding tensor
                (#batch, time2, size).
            cache (torch.Tensor): Cache tensor (1, head, cache_t, d_k * 2),
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        Returns:
            torch.Tensor: Output tensor (#batch, time1, d_model).
            torch.Tensor: Cache tensor (1, head, cache_t + time1, d_k * 2)
                where `cache_t == chunk_size * num_decoding_left_chunks`
                and `head * d_k == size`
        """
    q, k, v = self.forward_qkv(query, key, value)
    q = q.transpose(1, 2)  # (batch, time1, head, d_k)

    # NOTE(xcsong):
    #   when export onnx model, for 1st chunk, we feed
    #       cache(1, head, 0, d_k * 2) (16/-1, -1/-1, 16/0 mode)
    #       or cache(1, head, real_cache_t, d_k * 2) (16/4 mode).
    #       In all modes, `if cache.size(0) > 0` will alwayse be `True`
    #       and we will always do splitting and
    #       concatnation(this will simplify onnx export). Note that
    #       it's OK to concat & split zero-shaped tensors(see code below).
    #   when export jit  model, for 1st chunk, we always feed
    #       cache(0, 0, 0, 0) since jit supports dynamic if-branch.
    # >>> a = torch.ones((1, 2, 0, 4))
    # >>> b = torch.ones((1, 2, 3, 4))
    # >>> c = torch.cat((a, b), dim=2)
    # >>> torch.equal(b, c)        # True
    # >>> d = torch.split(a, 2, dim=-1)
    # >>> torch.equal(d[0], d[1])  # True
    if cache.size(0) > 0:
      key_cache, value_cache = torch.split(
        cache, cache.size(-1) // 2, dim=-1)
      k = torch.cat([key_cache, k], dim=2)
      v = torch.cat([value_cache, v], dim=2)
    # NOTE(xcsong): We do cache slicing in encoder.forward_chunk, since it's
    #   non-trivial to calculate `next_cache_start` here.
    new_cache = torch.cat((k, v), dim=-1)

    n_batch_pos = pos_emb.size(0)
    p = self.linear_pos(pos_emb).view(n_batch_pos, -1, self.h, self.d_k)
    p = p.transpose(1, 2)  # (batch, head, time1, d_k)

    # (batch, head, time1, d_k)
    q_with_bias_u = (q + self.pos_bias_u).transpose(1, 2)
    # (batch, head, time1, d_k)
    q_with_bias_v = (q + self.pos_bias_v).transpose(1, 2)

    # compute attention score
    # first compute matrix a and matrix c
    # as described in https://arxiv.org/abs/1901.02860 Section 3.3
    # (batch, head, time1, time2)
    matrix_ac = torch.matmul(q_with_bias_u, k.transpose(-2, -1))

    # compute matrix b and matrix d
    # (batch, head, time1, time2)
    matrix_bd = torch.matmul(q_with_bias_v, p.transpose(-2, -1))
    # Remove rel_shift since it is useless in speech recognition,
    # and it requires special attention for streaming.
    # matrix_bd = self.rel_shift(matrix_bd)

    scores = (matrix_ac + matrix_bd) / math.sqrt(
      self.d_k)  # (batch, head, time1, time2)

    return self.forward_attention(v, scores, mask), new_cache


class ConvolutionModule(nn.Module):
  """ConvolutionModule in Conformer model."""

  def __init__(self,
               channels: int,
               kernel_size: int = 15,
               activation: nn.Module = nn.ReLU(),
               norm: str = "batch_norm",
               causal: bool = False,
               bias: bool = True):
    """Construct an ConvolutionModule object.
        Args:
            channels (int): The number of channels of conv layers.
            kernel_size (int): Kernel size of conv layers.
            causal (int): Whether use causal convolution or not
        """
    assert check_argument_types()
    super().__init__()

    self.pointwise_conv1 = nn.Conv1d(
      channels,
      2 * channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    # self.lorder is used to distinguish if it's a causal convolution,
    # if self.lorder > 0: it's a causal convolution, the input will be
    #    padded with self.lorder frames on the left in forward.
    # else: it's a symmetrical convolution
    if causal:
      padding = 0
      self.lorder = kernel_size - 1
    else:
      # kernel_size should be an odd number for none causal convolution
      assert (kernel_size - 1) % 2 == 0
      padding = (kernel_size - 1) // 2
      self.lorder = 0
    self.depthwise_conv = nn.Conv1d(
      channels,
      channels,
      kernel_size,
      stride=1,
      padding=padding,
      groups=channels,
      bias=bias,
    )

    assert norm in ['batch_norm', 'layer_norm']
    if norm == "batch_norm":
      self.use_layer_norm = False
      self.norm = nn.BatchNorm1d(channels)
    else:
      self.use_layer_norm = True
      self.norm = nn.LayerNorm(channels)

    self.pointwise_conv2 = nn.Conv1d(
      channels,
      channels,
      kernel_size=1,
      stride=1,
      padding=0,
      bias=bias,
    )
    self.activation = activation

  def forward(
      self,
      x: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      cache: torch.Tensor = torch.zeros((0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute convolution module.
        Args:
            x (torch.Tensor): Input tensor (#batch, time, channels).
            mask_pad (torch.Tensor): used for batch padding (#batch, 1, time),
                (0, 0, 0) means fake mask.
            cache (torch.Tensor): left context cache, it is only
                used in causal convolution (#batch, channels, cache_t),
                (0, 0, 0) meas fake cache.
        Returns:
            torch.Tensor: Output tensor (#batch, time, channels).
        """
    # exchange the temporal dimension and the feature dimension
    x = x.transpose(1, 2)  # (#batch, channels, time)

    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    if self.lorder > 0:
      if cache.size(2) == 0:  # cache_t == 0
        x = nn.functional.pad(x, (self.lorder, 0), 'constant', 0.0)
      else:
        assert cache.size(0) == x.size(0)  # equal batch
        assert cache.size(1) == x.size(1)  # equal channel
        x = torch.cat((cache, x), dim=2)
      assert (x.size(2) > self.lorder)
      new_cache = x[:, :, -self.lorder:]
    else:
      # It's better we just return None if no cache is required,
      # However, for JIT export, here we just fake one tensor instead of
      # None.
      new_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)

    # GLU mechanism
    x = self.pointwise_conv1(x)  # (batch, 2*channel, dim)
    x = nn.functional.glu(x, dim=1)  # (batch, channel, dim)

    # 1D Depthwise Conv
    x = self.depthwise_conv(x)
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.activation(self.norm(x))
    if self.use_layer_norm:
      x = x.transpose(1, 2)
    x = self.pointwise_conv2(x)
    # mask batch padding
    if mask_pad.size(2) > 0:  # time > 0
      x.masked_fill_(~mask_pad, 0.0)

    return x.transpose(1, 2), new_cache


class PositionwiseFeedForward(torch.nn.Module):
  """Positionwise feed forward layer.

    FeedForward are appied on each position of the sequence.
    The output dim is same with the input dim.

    Args:
        idim (int): Input dimenstion.
        hidden_units (int): The number of hidden units.
        dropout_rate (float): Dropout rate.
        activation (torch.nn.Module): Activation function
    """

  def __init__(self,
               idim: int,
               hidden_units: int,
               dropout_rate: float,
               activation: torch.nn.Module = torch.nn.ReLU()):
    """Construct a PositionwiseFeedForward object."""
    super(PositionwiseFeedForward, self).__init__()
    self.w_1 = torch.nn.Linear(idim, hidden_units)
    self.activation = activation
    self.dropout = torch.nn.Dropout(dropout_rate)
    self.w_2 = torch.nn.Linear(hidden_units, idim)

  def forward(self, xs: torch.Tensor) -> torch.Tensor:
    """Forward function.

        Args:
            xs: input tensor (B, L, D)
        Returns:
            output tensor, (B, L, D)
        """
    return self.w_2(self.dropout(self.activation(self.w_1(xs))))


class ConformerEncoderLayer(torch.nn.Module):
  """Encoder layer module.
    Args:
        size (int): Input dimension.
        self_attn (torch.nn.Module): Self-attention module instance.
            `MultiHeadedAttention` or `RelPositionMultiHeadedAttention`
            instance can be used as the argument.
        feed_forward (torch.nn.Module): Feed-forward module instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        feed_forward_macaron (torch.nn.Module): Additional feed-forward module
             instance.
            `PositionwiseFeedForward` instance can be used as the argument.
        conv_module (torch.nn.Module): Convolution module instance.
            `ConvlutionModule` instance can be used as the argument.
        dropout_rate (float): Dropout rate.
        normalize_before (bool):
            True: use layer_norm before each sub-block.
            False: use layer_norm after each sub-block.
        concat_after (bool): Whether to concat attention layer's input and
            output.
            True: x -> x + linear(concat(x, att(x)))
            False: x -> x + att(x)
    """

  def __init__(
      self,
      size: int,
      self_attn: torch.nn.Module,
      feed_forward: Optional[nn.Module] = None,
      feed_forward_macaron: Optional[nn.Module] = None,
      conv_module: Optional[nn.Module] = None,
      dropout_rate: float = 0.1,
      normalize_before: bool = True,
      concat_after: bool = False,
  ):
    """Construct an EncoderLayer object."""
    super().__init__()
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.feed_forward_macaron = feed_forward_macaron
    self.conv_module = conv_module
    self.norm_ff = nn.LayerNorm(size, eps=1e-5)  # for the FNN module
    self.norm_mha = nn.LayerNorm(size, eps=1e-5)  # for the MHA module
    if feed_forward_macaron is not None:
      self.norm_ff_macaron = nn.LayerNorm(size, eps=1e-5)
      self.ff_scale = 0.5
    else:
      self.ff_scale = 1.0
    if self.conv_module is not None:
      self.norm_conv = nn.LayerNorm(size,
                                    eps=1e-5)  # for the CNN module
      self.norm_final = nn.LayerNorm(
        size, eps=1e-5)  # for the final output of the block
    self.dropout = nn.Dropout(dropout_rate)
    self.size = size
    self.normalize_before = normalize_before
    self.concat_after = concat_after
    if self.concat_after:
      self.concat_linear = nn.Linear(size + size, size)
    else:
      self.concat_linear = nn.Identity()

  def forward(
      self,
      x: torch.Tensor,
      mask: torch.Tensor,
      pos_emb: torch.Tensor,
      mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
      att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
      cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute encoded features.

        Args:
            x (torch.Tensor): (#batch, time, size)
            mask (torch.Tensor): Mask tensor for the input (#batch, time，time),
                (0, 0, 0) means fake mask.
            pos_emb (torch.Tensor): positional encoding, must not be None
                for ConformerEncoderLayer.
            mask_pad (torch.Tensor): batch padding mask used for conv module.
                (#batch, 1，time), (0, 0, 0) means fake mask.
            att_cache (torch.Tensor): Cache tensor of the KEY & VALUE
                (#batch=1, head, cache_t1, d_k * 2), head * d_k == size.
            cnn_cache (torch.Tensor): Convolution cache in conformer layer
                (#batch=1, size, cache_t2)
        Returns:
            torch.Tensor: Output tensor (#batch, time, size).
            torch.Tensor: Mask tensor (#batch, time, time).
            torch.Tensor: att_cache tensor,
                (#batch=1, head, cache_t1 + time, d_k * 2).
            torch.Tensor: cnn_cahce tensor (#batch, size, cache_t2).
        """

    # whether to use macaron style
    if self.feed_forward_macaron is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_ff_macaron(x)
      x = residual + self.ff_scale * self.dropout(
        self.feed_forward_macaron(x))
      if not self.normalize_before:
        x = self.norm_ff_macaron(x)

    # multi-headed self-attention module
    residual = x
    if self.normalize_before:
      x = self.norm_mha(x)

    x_att, new_att_cache = self.self_attn(
      x, x, x, mask, pos_emb, att_cache)
    if self.concat_after:
      x_concat = torch.cat((x, x_att), dim=-1)
      x = residual + self.concat_linear(x_concat)
    else:
      x = residual + self.dropout(x_att)
    if not self.normalize_before:
      x = self.norm_mha(x)

    # convolution module
    # Fake new cnn cache here, and then change it in conv_module
    new_cnn_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)
    if self.conv_module is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_conv(x)
      x, new_cnn_cache = self.conv_module(x, mask_pad, cnn_cache)
      x = residual + self.dropout(x)

      if not self.normalize_before:
        x = self.norm_conv(x)

    # feed forward module
    residual = x
    if self.normalize_before:
      x = self.norm_ff(x)

    x = residual + self.ff_scale * self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x = self.norm_ff(x)

    if self.conv_module is not None:
      x = self.norm_final(x)

    return x, mask, new_att_cache, new_cnn_cache


class ConformerEncoder(torch.nn.Module):
  def __init__(
      self,
      input_size: int,
      output_size: int,
      attention_heads: int = 4,
      linear_units: int = 2048,
      num_blocks: int = 6,
      dropout_rate: float = 0.1,
      positional_dropout_rate: float = 0.1,
      attention_dropout_rate: float = 0.0,
      input_layer: str = "conv2d",
      pos_enc_layer_type: str = "rel_pos",
      normalize_before: bool = True,
      concat_after: bool = False,
      static_chunk_size: int = 0,
      use_dynamic_chunk: bool = False,
      global_cmvn: Optional[torch.nn.Module] = None,
      use_dynamic_left_chunk: bool = False,
      positionwise_conv_kernel_size: int = 1,
      macaron_style: bool = True,
      selfattention_layer_type: str = "rel_selfattn",
      activation_type: str = "swish",
      use_cnn_module: bool = True,
      cnn_module_kernel: int = 15,
      causal: bool = False,
      cnn_module_norm: str = "batch_norm",
  ):
    """
        Args:
            input_size (int): input dim
            output_size (int): dimension of attention
            attention_heads (int): the number of heads of multi head attention
            linear_units (int): the hidden units number of position-wise feed
                forward
            num_blocks (int): the number of decoder blocks
            dropout_rate (float): dropout rate
            attention_dropout_rate (float): dropout rate in attention
            positional_dropout_rate (float): dropout rate after adding
                positional encoding
            input_layer (str): input layer type.
                optional [linear, conv2d, conv2d6, conv2d8]
            pos_enc_layer_type (str): Encoder positional encoding layer type.
                opitonal [abs_pos, scaled_abs_pos, rel_pos, no_pos]
            normalize_before (bool):
                True: use layer_norm before each sub-block of a layer.
                False: use layer_norm after each sub-block of a layer.
            concat_after (bool): whether to concat attention layer's input
                and output.
                True: x -> x + linear(concat(x, att(x)))
                False: x -> x + att(x)
            static_chunk_size (int): chunk size for static chunk training and
                decoding
            use_dynamic_chunk (bool): whether use dynamic chunk size for
                training or not, You can only use fixed chunk(chunk_size > 0)
                or dyanmic chunk size(use_dynamic_chunk = True)
            global_cmvn (Optional[torch.nn.Module]): Optional GlobalCMVN module
            use_dynamic_left_chunk (bool): whether use dynamic left chunk in
                dynamic chunk training
        """
    assert check_argument_types()
    logging.info("Init Conformer layers::")
    super().__init__()
    self._output_size = output_size

    if pos_enc_layer_type == "abs_pos":
      pos_enc_class = PositionalEncoding
    elif pos_enc_layer_type == "rel_pos":
      pos_enc_class = RelPositionalEncoding
    elif pos_enc_layer_type == "no_pos":
      pos_enc_class = NoPositionalEncoding
    else:
      raise ValueError("unknown pos_enc_layer: " + pos_enc_layer_type)
    logging.info("Pos layers - {}".format(pos_enc_layer_type))

    if input_layer == "linear":
      subsampling_class = LinearNoSubsampling
    elif input_layer == "conv2d":
      subsampling_class = Conv2dSubsampling4
    elif input_layer == "conv2d8":
      subsampling_class = Conv2dSubsampling8
    else:
      raise ValueError("unknown input_layer: " + input_layer)
    logging.info("Input layers - {}".format(input_layer))

    self.global_cmvn = global_cmvn
    self.embed = subsampling_class(
      input_size,
      output_size,
      dropout_rate,
      pos_enc_class(output_size, positional_dropout_rate),
    )

    self.normalize_before = normalize_before
    self.after_norm = torch.nn.LayerNorm(output_size, eps=1e-5)
    self.static_chunk_size = static_chunk_size
    self.use_dynamic_chunk = use_dynamic_chunk
    self.use_dynamic_left_chunk = use_dynamic_left_chunk

    activation = get_activation(activation_type)

    # self-attention module definition
    if pos_enc_layer_type != "rel_pos":
      encoder_selfattn_layer = MultiHeadedAttention
    else:
      encoder_selfattn_layer = RelPositionMultiHeadedAttention
    encoder_selfattn_layer_args = (
      attention_heads,
      output_size,
      attention_dropout_rate,
    )
    logging.info("self attn args: {}".format(encoder_selfattn_layer_args))

    # feed-forward module definition
    positionwise_layer = PositionwiseFeedForward
    positionwise_layer_args = (
      output_size,
      linear_units,
      dropout_rate,
      activation,
    )
    logging.info("position-wise args: {}".format(positionwise_layer_args))

    # convolution module definition
    convolution_layer = ConvolutionModule
    convolution_layer_args = (output_size, cnn_module_kernel, activation,
                              cnn_module_norm, causal)

    self.encoders = torch.nn.ModuleList([
      ConformerEncoderLayer(
        output_size,
        encoder_selfattn_layer(*encoder_selfattn_layer_args),
        positionwise_layer(*positionwise_layer_args),
        positionwise_layer(
          *positionwise_layer_args) if macaron_style else None,
        convolution_layer(
          *convolution_layer_args) if use_cnn_module else None,
        dropout_rate,
        normalize_before,
        concat_after,
      ) for _ in range(num_blocks)
    ])
    return

  def output_size(self) -> int:
    return self._output_size

  def forward(
      self,
      xs: torch.Tensor,
      xs_lens: torch.Tensor,
      decoding_chunk_size: int = -1,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Embed positions in tensor.

        Args:
            xs: padded input tensor (B, T, D)
            xs_lens: input length (B)
            decoding_chunk_size: decoding chunk size for dynamic chunk
                0: default for training, use random dynamic chunk.
                <0: for decoding, use full chunk.
                >0: for decoding, use fixed chunk size as set.
            num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
                >=0: use num_decoding_left_chunks
                <0: use all left chunks
        Returns:
            encoder output tensor xs, and subsampled masks
            xs: padded output tensor (B, T' ~= T/subsample_rate, D)
            masks: torch.Tensor batch padding mask after subsample
                (B, 1, T' ~= T/subsample_rate)
        """
    T = xs.size(1)
    masks = ~make_pad_mask(xs_lens, T).unsqueeze(1)  # (B, 1, T)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    xs, pos_emb, masks = self.embed(xs, masks)
    mask_pad = masks  # (B, 1, T/subsample_rate)
    chunk_masks = add_optional_chunk_mask(xs, masks,
                                          self.use_dynamic_chunk,
                                          self.use_dynamic_left_chunk,
                                          decoding_chunk_size,
                                          self.static_chunk_size,
                                          num_decoding_left_chunks)
    for layer in self.encoders:
      xs, chunk_masks, _, _ = layer(xs, chunk_masks, pos_emb, mask_pad)
    if self.normalize_before:
      xs = self.after_norm(xs)
    # Here we assume the mask is not changed in encoder layers, so just
    # return the masks before encoder layers, and the masks will be used
    # for cross attention with decoder later
    return xs, masks

  def forward_chunk(
      self,
      xs: torch.Tensor,
      offset: int,
      required_cache_size: int,
      att_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      cnn_cache: torch.Tensor = torch.zeros(0, 0, 0, 0),
      att_mask: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """ Forward just one chunk

        Args:
            xs (torch.Tensor): chunk input, with shape (b=1, time, mel-dim),
                where `time == (chunk_size - 1) * subsample_rate + \
                        subsample.right_context + 1`
            offset (int): current offset in encoder output time stamp
            required_cache_size (int): cache size required for next chunk
                compuation
                >=0: actual cache size
                <0: means all history cache is required
            att_cache (torch.Tensor): cache tensor for KEY & VALUE in
                transformer/conformer attention, with shape
                (elayers, head, cache_t1, d_k * 2), where
                `head * d_k == hidden-dim` and
                `cache_t1 == chunk_size * num_decoding_left_chunks`.
            cnn_cache (torch.Tensor): cache tensor for cnn_module in conformer,
                (elayers, b=1, hidden-dim, cache_t2), where
                `cache_t2 == cnn.lorder - 1`

        Returns:
            torch.Tensor: output of current input xs,
                with shape (b=1, chunk_size, hidden-dim).
            torch.Tensor: new attention cache required for next chunk, with
                dynamic shape (elayers, head, ?, d_k * 2)
                depending on required_cache_size.
            torch.Tensor: new conformer cnn cache required for next chunk, with
                same shape as the original cnn_cache.

        """
    assert xs.size(0) == 1
    # tmp_masks is just for interface compatibility
    tmp_masks = torch.ones(1,
                           xs.size(1),
                           device=xs.device,
                           dtype=torch.bool)
    tmp_masks = tmp_masks.unsqueeze(1)
    if self.global_cmvn is not None:
      xs = self.global_cmvn(xs)
    # NOTE(xcsong): Before embed, shape(xs) is (b=1, time, mel-dim)
    xs, pos_emb, _ = self.embed(xs, tmp_masks, offset)
    # NOTE(xcsong): After  embed, shape(xs) is (b=1, chunk_size, hidden-dim)
    elayers, cache_t1 = att_cache.size(0), att_cache.size(2)
    chunk_size = xs.size(1)
    attention_key_size = cache_t1 + chunk_size
    pos_emb = self.embed.position_encoding(
      offset=offset - cache_t1, size=attention_key_size)
    if required_cache_size < 0:
      next_cache_start = 0
    elif required_cache_size == 0:
      next_cache_start = attention_key_size
    else:
      next_cache_start = max(attention_key_size - required_cache_size, 0)
    r_att_cache = []
    r_cnn_cache = []
    for i, layer in enumerate(self.encoders):
      # NOTE(xcsong): Before layer.forward
      #   shape(att_cache[i:i + 1]) is (1, head, cache_t1, d_k * 2),
      #   shape(cnn_cache[i])       is (b=1, hidden-dim, cache_t2)
      xs, _, new_att_cache, new_cnn_cache = layer(
        xs, att_mask, pos_emb,
        att_cache=att_cache[i:i + 1] if elayers > 0 else att_cache,
        cnn_cache=cnn_cache[i] if cnn_cache.size(0) > 0 else cnn_cache
      )
      # NOTE(xcsong): After layer.forward
      #   shape(new_att_cache) is (1, head, attention_key_size, d_k * 2),
      #   shape(new_cnn_cache) is (b=1, hidden-dim, cache_t2)
      r_att_cache.append(new_att_cache[:, :, next_cache_start:, :])
      r_cnn_cache.append(new_cnn_cache.unsqueeze(0))
    if self.normalize_before:
      xs = self.after_norm(xs)

    # NOTE(xcsong): shape(r_att_cache) is (elayers, head, ?, d_k * 2),
    #   ? may be larger than cache_t1, it depends on required_cache_size
    r_att_cache = torch.cat(r_att_cache, dim=0)
    # NOTE(xcsong): shape(r_cnn_cache) is (e, b=1, hidden-dim, cache_t2)
    r_cnn_cache = torch.cat(r_cnn_cache, dim=0)

    return (xs, r_att_cache, r_cnn_cache)

  def forward_chunk_by_chunk(
      self,
      xs: torch.Tensor,
      decoding_chunk_size: int,
      num_decoding_left_chunks: int = -1,
  ) -> Tuple[torch.Tensor, torch.Tensor]:
    """ Forward input chunk by chunk with chunk_size like a streaming
            fashion

        Here we should pay special attention to computation cache in the
        streaming style forward chunk by chunk. Three things should be taken
        into account for computation in the current network:
            1. transformer/conformer encoder layers output cache
            2. convolution in conformer
            3. convolution in subsampling

        However, we don't implement subsampling cache for:
            1. We can control subsampling module to output the right result by
               overlapping input instead of cache left context, even though it
               wastes some computation, but subsampling only takes a very
               small fraction of computation in the whole model.
            2. Typically, there are several covolution layers with subsampling
               in subsampling module, it is tricky and complicated to do cache
               with different convolution layers with different subsampling
               rate.
            3. Currently, nn.Sequential is used to stack all the convolution
               layers in subsampling, we need to rewrite it to make it work
               with cache, which is not prefered.
        Args:
            xs (torch.Tensor): (1, max_len, dim)
            chunk_size (int): decoding chunk size
        """
    assert decoding_chunk_size > 0
    # The model is trained by static or dynamic chunk
    assert self.static_chunk_size > 0 or self.use_dynamic_chunk
    subsampling = self.embed.subsampling_rate
    context = self.embed.right_context + 1  # Add current frame
    stride = subsampling * decoding_chunk_size
    decoding_window = (decoding_chunk_size - 1) * subsampling + context
    num_frames = xs.size(1)
    att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0), device=xs.device)
    outputs = []
    offset = 0
    required_cache_size = decoding_chunk_size * num_decoding_left_chunks

    # Feed forward overlap input step by step
    for cur in range(0, num_frames - context + 1, stride):
      end = min(cur + decoding_window, num_frames)
      chunk_xs = xs[:, cur:end, :]
      (y, att_cache, cnn_cache) = self.forward_chunk(
        chunk_xs, offset, required_cache_size, att_cache, cnn_cache)
      outputs.append(y)
      offset += y.size(1)
    ys = torch.cat(outputs, 1)
    masks = torch.ones((1, 1, ys.size(1)), device=ys.device, dtype=torch.bool)
    return ys, masks


class AttentiveStatisticsPooling(torch.nn.Module):
  """This class implements an attentive statistic pooling layer for each channel.
  It returns the concatenated mean and std of the input tensor.

  Arguments
  ---------
  channels: int
      The number of input channels.
  output_channels: int
      The number of output channels.
  """

  def __init__(self, channels, output_channels):
    super().__init__()

    self._eps = 1e-12
    self._linear = Linear(channels * 3, channels)
    self._tanh = torch.nn.Tanh()
    self._conv = Conv1d(
      in_channels=channels, out_channels=channels, kernel_size=1
    )
    self._final_layer = torch.nn.Linear(channels * 2, output_channels,
                                        bias=False)
    logging.info("Init AttentiveStatisticsPooling with {}->{}".format(
      channels, output_channels))
    return

  @staticmethod
  def _compute_statistics(x: torch.Tensor,
                          m: torch.Tensor,
                          eps: float,
                          dim: int = 2):
    mean = (m * x).sum(dim)
    std = torch.sqrt((m * (x - mean.unsqueeze(dim)).pow(2)).sum(dim).clamp(eps))
    return mean, std

  def forward(self, x: torch.Tensor):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, L, C].
    """

    x = x.transpose(1, 2)
    L = x.shape[-1]
    lengths = torch.ones(x.shape[0], device=x.device)
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)
    total = mask.sum(dim=2, keepdim=True).float()

    mean, std = self._compute_statistics(x, mask / total, self._eps)
    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)
    attn = self._conv(self._tanh(self._linear(
      attn.transpose(1, 2)).transpose(1, 2)))

    attn = attn.masked_fill(mask == 0, float("-inf"))  # Filter out zero-padding
    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    pooled_stats = self._final_layer(torch.cat((mean, std), dim=1))
    return pooled_stats

  def forward_with_mask(self, x: torch.Tensor,
                        lengths: Optional[torch.Tensor] = None):
    """Calculates mean and std for a batch (input tensor).

    Args:
      x : torch.Tensor
          Tensor of shape [N, C, L].
      lengths:
    """
    L = x.shape[-1]

    if lengths is None:
      lengths = torch.ones(x.shape[0], device=x.device)

    # Make binary mask of shape [N, 1, L]
    mask = self.length_to_mask(lengths * L, max_len=L, device=x.device)
    mask = mask.unsqueeze(1)

    # Expand the temporal context of the pooling layer by allowing the
    # self-attention to look at global properties of the utterance.

    # torch.std is unstable for backward computation
    # https://github.com/pytorch/pytorch/issues/4320
    total = mask.sum(dim=2, keepdim=True).float()
    mean, std = self._compute_statistics(x, mask / total, self._eps)

    mean = mean.unsqueeze(2).repeat(1, 1, L)
    std = std.unsqueeze(2).repeat(1, 1, L)
    attn = torch.cat([x, mean, std], dim=1)

    # Apply layers
    attn = self.conv(self._tanh(self._linear(attn, lengths)))

    # Filter out zero-paddings
    attn = attn.masked_fill(mask == 0, float("-inf"))

    attn = F.softmax(attn, dim=2)
    mean, std = self._compute_statistics(x, attn, self._eps)
    # Append mean and std of the batch
    pooled_stats = torch.cat((mean, std), dim=1)
    pooled_stats = pooled_stats.unsqueeze(2)
    return pooled_stats

  @staticmethod
  def length_to_mask(length: torch.Tensor,
                     max_len: Optional[int] = None,
                     dtype: Optional[torch.dtype] = None,
                     device: Optional[torch.device] = None):
    """Creates a binary mask for each sequence.

    Arguments
    ---------
    length : torch.LongTensor
        Containing the length of each sequence in the batch. Must be 1D.
    max_len : int
        Max length for the mask, also the size of the second dimension.
    dtype : torch.dtype, default: None
        The dtype of the generated mask.
    device: torch.device, default: None
        The device to put the mask variable.

    Returns
    -------
    mask : tensor
        The binary mask.

    Example
    -------
    """
    assert len(length.shape) == 1

    if max_len is None:
      max_len = length.max().long().item()  # using arange to generate mask
    mask = torch.arange(
      max_len, device=length.device, dtype=length.dtype
    ).expand(len(length), max_len) < length.unsqueeze(1)

    if dtype is None:
      dtype = length.dtype

    if device is None:
      device = length.device

    mask = torch.as_tensor(mask, dtype=dtype, device=device)
    return mask


class Model(BasicModel):
  """ Csi Backbone Model:
      Batch-norm
      Conformer-Encoder(x6 or x4)
      Global-Avg-Pool
      Resize and linear
      Bn-neck

  Loss:
    Focal-loss(ce) + Triplet-loss + Center-loss

  Args:
    hp: Dict with all parameters
  """

  def __init__(self, hp: Dict):
    super(BasicModel, self).__init__()
    self._hp = hp

    self._global_cmvn = torch.nn.BatchNorm1d(hp["input_dim"])
    self._encoder = ConformerEncoder(
      input_size=hp["input_dim"],
      output_size=hp["encoder"]["output_dims"],
      linear_units=hp["encoder"]["attention_dim"],
      num_blocks=hp["encoder"]["num_blocks"])

    if hp["encoder"]["output_dims"] != hp["embed_dim"]:
      self._embed_lo = torch.nn.Linear(hp["encoder"]["output_dims"],
                                       hp["embed_dim"])
    else:
      self._embed_lo = None

    # Bottleneck
    self._bottleneck = torch.nn.BatchNorm1d(hp["embed_dim"])
    self._bottleneck.bias.requires_grad_(False)  # no shift

    self._pool_layer = AttentiveStatisticsPooling(
      hp["embed_dim"], output_channels=hp["embed_dim"])
    self._ce_layer = torch.nn.Linear(
      hp["embed_dim"], hp["ce"]["output_dims"], bias=False)

    # Loss
    if "alpha" in hp["ce"].keys():
      alpha = np.load(hp["ce"]["alpha"])
      alpha = 1.0 / (alpha + 1)
      alpha = alpha / np.sum(alpha)
      logging.warning("use alpha with {}".format(len(alpha)))
    else:
      alpha = None
      logging.warning("Not use alpha")

    self._ce_loss = FocalLoss(alpha=alpha, gamma=self._hp["ce"]["gamma"],
                              num_cls=self._hp["ce"]["output_dims"])
    self._triplet_loss = HardTripletLoss(margin=hp["triplet"]["margin"])
    self._center_loss = CenterLoss(num_classes=self._hp["ce"]["output_dims"],
                                   feat_dim=hp["embed_dim"], use_gpu=torch.cuda.is_available())
    logging.info(
      "Model size: {:.3f}M \n".format(self.model_size() / 1000 / 1000))
    return

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """feat[b, frame_size, feat_size] -> embed[b, embed_dim]"""

    x = self._global_cmvn(x.transpose(1, 2)).transpose(1, 2)
    xs_lens = torch.full(
      [x.size(0)], fill_value=x.size(1), dtype=torch.long).to(x.device).long()
    x, _ = self._encoder(x, xs_lens=xs_lens, decoding_chunk_size=-1)
    x = self._pool_layer(x)
    return x

  @torch.jit.ignore
  def compute_loss(self, anchor: torch.Tensor, label: torch.Tensor) -> Tuple[
    torch.Tensor, Dict]:
    """compute ce and triplet loss"""
    f_t = self.forward(anchor)
    # print("f_t", f_t)

    f_i = self._bottleneck(f_t)
    ce_pred = self._ce_layer(f_i)
    # print("ce_pred", ce_pred)
    ce_loss = self._ce_loss(ce_pred, label)
    loss = ce_loss * self._hp["ce"]["weight"]
    loss_dict = {"ce_loss": ce_loss}

    tri_weight = self._hp["triplet"]["weight"]
    if tri_weight > 0.01:
      tri_loss = self._triplet_loss(f_t, label)
      loss = loss + tri_loss * tri_weight
      loss_dict.update({"tri_loss": tri_loss})

    cen_weight = self._hp["center"]["weight"]
    if cen_weight > 0.0:
      cen_loss = self._center_loss(f_t, label)
      loss = loss + cen_loss * cen_weight
      loss_dict.update({"cen_loss": cen_loss})
    return loss, loss_dict

  @torch.jit.ignore
  def inference(self, feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    with torch.no_grad():
      embed = self.forward(feat)
      embed_ce = self._ce_layer(embed)
    return embed, embed_ce

  @torch.jit.export
  def compute_embed(self, feat: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
      embed = self.forward(feat)
    return embed


### 2.1 Lop tien xu ly tin hieu am thanh (AudioPreprocessor)
Resample am thanh ve 16,000 Hz, chuan hoa bien do cuc dai ve [-1.0, 1.0], va cat thanh cac cua so truot 10.0 giay voi buoc nhay 3.0 giay.

In [ ]:
class AudioPreprocessor:
    """Tiền xử lý âm thanh: Resample về 16,000 Hz, chuẩn hóa biên độ và cắt Sliding Window 10s / hop 3s."""
    def __init__(self, sample_rate: int = 16000, chunk_duration: float = 10.0, hop_duration: float = 3.0):
        self.sample_rate = sample_rate
        self.chunk_duration = chunk_duration
        self.hop_duration = hop_duration
        self.chunk_samples = int(chunk_duration * sample_rate)
        self.hop_samples = int(hop_duration * sample_rate)

    def load_audio(self, audio_path: str) -> np.ndarray:
        """Nạp và chuẩn hóa âm thanh, không nuốt lỗi âm thầm."""
        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Không tìm thấy file âm thanh: {audio_path}")
            
        if audio_path.lower().endswith('.silent'):
            return np.zeros(self.chunk_samples, dtype=np.float32)

        try:
            if audio_path.lower().endswith('.mp3'):
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            else:
                y, sr = sf.read(audio_path, always_2d=False, dtype='float32')
                if len(y.shape) > 1:
                    y = np.mean(y, axis=1)
                if sr != self.sample_rate:
                    y = librosa.resample(y, orig_sr=sr, target_sr=self.sample_rate)
        except Exception as e:
            # Fallback thu doc bang librosa neu soundfile gap loi dinh dang wav
            try:
                y, sr = librosa.load(audio_path, sr=self.sample_rate, duration=None)
            except Exception as final_e:
                print(f"[!] Canh bao: Khong the doc file {audio_path}: {final_e}")
                raise RuntimeError(f"Không thể đọc file âm thanh: {audio_path}") from final_e

        y = np.asarray(y, dtype=np.float32).reshape(-1)
        if y.size == 0:
            raise ValueError(f"File audio rong: {audio_path}")
        if not np.isfinite(y).all():
            raise ValueError(f"Audio chua NaN/Inf: {audio_path}")
        max_val = float(np.max(np.abs(y)))
        if max_val > 1e-6:
            y = y / max_val
        return y

    def slice_into_chunks(self, y: np.ndarray) -> list:
        """Cắt âm thanh thành danh sách các đoạn 10 giây (hop 3s). Trả về ít nhất 1 đoạn."""
        if len(y) <= self.chunk_samples:
            return [np.pad(y, (0, self.chunk_samples - len(y)), mode='constant')]
        chunks = []
        start = 0
        for start in range(0, len(y) - self.chunk_samples + 1, self.hop_samples):
            chunks.append(y[start : start + self.chunk_samples])
        last_chunk_end = start + self.chunk_samples
        remaining_unseen = len(y) - last_chunk_end
        if remaining_unseen > (self.chunk_samples // 3):
            final_chunk = y[-self.chunk_samples:]
            if not np.array_equal(final_chunk, chunks[-1]):
                chunks.append(final_chunk)
        return chunks if chunks else [np.pad(y, (0, max(0, self.chunk_samples - len(y))), mode='constant')[:self.chunk_samples]]

### 2.2 Lop trich xuat dac trung pho Constant-Q Transform (CoverHunterCQT)
Trich xuat pho CQT 96 bins tan so (12 bins/octave, 8 quang tam tu 32 Hz den 8000 Hz), hop 0.04s, nen thoi gian qua ham shorter(mean_size=3) va chuan hoa dB theo dung chuan CoverHunter goc.

In [ ]:
import scipy.sparse

def shorter(feat: np.ndarray, mean_size: int = 3) -> np.ndarray:
    """Nén trục thời gian của ma trận đặc trưng CQT theo bước mean_size (CoverHunter chuẩn)."""
    if mean_size <= 1:
        return feat
    cqt = feat.T
    height, length = cqt.shape
    new_len = int(length / mean_size)
    new_cqt = np.zeros((height, new_len), dtype=np.float32)
    for i in range(new_len):
        new_cqt[:, i] = cqt[:, i * mean_size : (i + 1) * mean_size].mean(axis=1)
    return new_cqt.T

class PyCqt:
    """Thuật toán CQT Sparse Kernel chính thức của CoverHunter (Tsinghua/MIREX)."""
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = None):
        self._hop_size = hop_size
        self._sample_rate = sample_rate
        if not max_freq:
            max_freq = sample_rate // 2
        self._kernel = self._compute_cqt_kernel(sample_rate, octave_resolution, min_freq, max_freq)

    @staticmethod
    def _compute_cqt_kernel(sampling_frequency: int, octave_resolution: int, minimum_frequency: float, maximum_frequency: float):
        quality_factor = 1.0 / (pow(2.0, 1.0 / octave_resolution) - 1.0)
        number_frequencies = round(octave_resolution * np.log2(maximum_frequency / minimum_frequency))
        fft_length = int(pow(2, np.ceil(np.log2(quality_factor * sampling_frequency / minimum_frequency))))
        cqt_kernel = np.zeros((number_frequencies, fft_length), dtype=complex)

        for i in range(number_frequencies):
            frequency_value = minimum_frequency * pow(2.0, i / octave_resolution)
            window_length = 2 * round(quality_factor * sampling_frequency / frequency_value / 2.0) + 1
            temporal_kernel = (np.hamming(window_length)
                               * np.exp(2.0 * np.pi * 1j * quality_factor
                                        * np.arange(-(window_length - 1) / 2.0, (window_length - 1) / 2.0 + 1) / window_length)
                               / window_length)
            pad_width = int((fft_length - window_length + 1) / 2)
            cqt_kernel[i, pad_width : pad_width + window_length] = temporal_kernel

        cqt_kernel = np.fft.fft(cqt_kernel, axis=1)
        cqt_kernel[np.absolute(cqt_kernel) < 0.01] = 0
        cqt_kernel = scipy.sparse.csr_matrix(cqt_kernel)
        cqt_kernel = np.conjugate(cqt_kernel) / fft_length
        return cqt_kernel

    @staticmethod
    def _compute_cqt_spec(audio_signal: np.ndarray, sampling_frequency: int, time_resolution: int, cqt_kernel):
        step_length = round(sampling_frequency / time_resolution)
        number_times = int(np.floor(len(audio_signal) / step_length))
        number_frequencies, fft_length = np.shape(cqt_kernel)

        pad_left = int(np.ceil((fft_length - step_length) / 2.0))
        pad_right = int(np.floor((fft_length - step_length) / 2.0))
        audio_signal = np.pad(audio_signal, (pad_left, pad_right), "constant", constant_values=(0, 0))

        cqt_spectrogram = np.zeros((number_frequencies, number_times), dtype=np.float32)
        idx = 0
        for j in range(number_times):
            cqt_spectrogram[:, j] = np.absolute(cqt_kernel * np.fft.fft(audio_signal[idx : idx + fft_length]))
            idx += step_length
        return cqt_spectrogram

    def compute_cqt(self, signal_float: np.ndarray, feat_dim_first: bool = False) -> np.ndarray:
        time_resolution = int(round(1.0 / self._hop_size))
        cqt_spectrogram = self._compute_cqt_spec(signal_float, self._sample_rate, time_resolution, self._kernel)
        magnitude = np.maximum(cqt_spectrogram, 1e-9)
        ref_value = max(float(np.max(magnitude)), 1e-4)
        cqt_spectrogram = 20.0 * np.log10(magnitude) - 20.0 * np.log10(ref_value)
        cqt_spectrogram = np.clip(cqt_spectrogram, -80.0, 0.0)
        if not feat_dim_first:
            cqt_spectrogram = cqt_spectrogram.T
        return cqt_spectrogram

class CoverHunterCQT:
    """
    Trích xuất đặc trưng CQT chuẩn CoverHunter:
    - 96 bins (12 bins/octave, 8 octaves từ 32 Hz đến 8000 Hz)
    - hop_size: 0.04s (25 fps), nén thời gian shorter(mean_size=3)
    - Chuẩn hóa dB theo biên độ cực đại: 20 * log10(cqt / max_cqt)
    """
    def __init__(self, sample_rate: int = 16000, hop_size: float = 0.04, octave_resolution: int = 12, min_freq: float = 32.0, max_freq: float = 8000.0, mean_size: int = 3):
        self.sample_rate = sample_rate
        self.mean_size = mean_size
        self.py_cqt = PyCqt(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=octave_resolution,
            min_freq=min_freq,
            max_freq=max_freq
        )

    def extract(self, audio: np.ndarray) -> torch.Tensor:
        if isinstance(audio, torch.Tensor):
            audio = audio.detach().cpu().numpy()
        feat = self.py_cqt.compute_cqt(audio, feat_dim_first=False) # (T, 96)
        if self.mean_size > 1:
            feat = shorter(feat, mean_size=self.mean_size)
        return torch.from_numpy(feat).float()

### 2.3 Lop can chinh kich thuoc pho thoi gian (CQTExtractor)
Ho tro 2 che do trich xuat: do dai co dinh 83 frames (10s) phuc vu training batch va FAISS global embedding, hoac do dai dong (variable_length=True) giu nguyen so frames thuc te phuc vu can chinh DTW.

In [ ]:
class CQTExtractor:
    """Trích xuất Constant-Q Transform (CQT) hỗ trợ cả chế độ Fixed-Length (83 frames) và Variable-Length."""
    def __init__(self, sample_rate: int = 16000, n_bins: int = 96, bins_per_octave: int = 12, hop_size: float = 0.04, mean_size: int = 3, target_frames: int = 83):
        self.sample_rate = sample_rate
        self.n_bins = n_bins
        self.bins_per_octave = bins_per_octave
        self.hop_size = hop_size
        self.mean_size = mean_size
        self.target_frames = target_frames
        self.cqt_impl = CoverHunterCQT(
            sample_rate=sample_rate,
            hop_size=hop_size,
            octave_resolution=bins_per_octave,
            min_freq=32.0,
            max_freq=8000.0,
            mean_size=mean_size
        )

    def extract(self, y: np.ndarray, variable_length: bool = False) -> torch.Tensor:
        """
        Trích xuất tensor CQT từ tín hiệu âm thanh 1D.
        - variable_length=False: Căn chỉnh về đúng [1, 96, target_frames] (mặc định 83 frames ~ 10s).
        - variable_length=True: Giữ nguyên độ dài thời gian thực tế [1, 96, T] phục vụ Stage 2 DTW.
        """
        feat = self.cqt_impl.extract(y)  # (T, 96)
        tensor = feat.T.unsqueeze(0)      # (1, 96, T)

        if not variable_length and self.target_frames is not None and self.target_frames > 0:
            curr_t = tensor.shape[-1]
            min_val = tensor.min()
            if curr_t < self.target_frames:
                tensor = F.pad(tensor, (0, self.target_frames - curr_t), mode='constant', value=min_val.item())
            elif curr_t > self.target_frames:
                tensor = tensor[:, :, :self.target_frames]
        return tensor

SAMPLE_RATE = 16000
CHUNK_DURATION_SEC = 10.0
CHUNK_HOP_DURATION_SEC = 3.0

prep = AudioPreprocessor(sample_rate=SAMPLE_RATE, chunk_duration=CHUNK_DURATION_SEC, hop_duration=CHUNK_HOP_DURATION_SEC)
cqt_ext = CQTExtractor(sample_rate=16000, n_bins=96, bins_per_octave=12, hop_size=0.04, mean_size=3, target_frames=83)
print("[+] Khởi tạo thành công AudioPreprocessor và CoverHunter CQT Standalone (96 Bins, 83 Frames, dB Scale)!")

### 3.1 Ham dich cao do bat bien quang tam (apply_cqt_pitch_shift)
Dich chuyen truc tan so cua CQT de mo phong nguoi hat lech tone. Khac phuc loi am hoc: dien cac bin trong bang muc san nhieu `cqt.min()` (khoang lang -140 dB), tuyet doi khong dien 0.0.

In [ ]:
def apply_cqt_pitch_shift(cqt: torch.Tensor, shift_bins: int) -> torch.Tensor:
    """
    Dịch chuyển cao độ trên ma trận CQT bằng cách tịnh tiến trục tần số (dim -2).
    - shift_bins > 0: Dịch lên cao độ cao hơn (tăng semitone).
    - shift_bins < 0: Dịch xuống cao độ thấp hơn (giảm semitone).
    - Điền các bin trống bằng cqt.min() (mức sàn im lặng), KHÔNG điền 0.0 (vốn là 0 dB cực đại).
    """
    if shift_bins == 0:
        return cqt

    min_val = cqt.min()
    shifted = torch.full_like(cqt, fill_value=min_val)
    num_bins = cqt.shape[-2]

    if shift_bins > 0:
        if shift_bins < num_bins:
            shifted[..., shift_bins:, :] = cqt[..., :num_bins - shift_bins, :]
    else:
        abs_shift = abs(shift_bins)
        if abs_shift < num_bins:
            shifted[..., :num_bins - abs_shift, :] = cqt[..., abs_shift:, :]

    return shifted

### 3.2 Lop tang cuong dich tone ngau nhien (CQTPitchShiftAugmentation)
Dich ngau nhien trong dai [-12, +12] semitones tron 1 quang 8 (xac suat 80%), giup mo hinh bat bien hoan toan truoc su khac biet cao do giua giong Nam va giong Nu.

In [ ]:
class CQTPitchShiftAugmentation:
    """
    Tăng cường dữ liệu dịch tone ngẫu nhiên (Full-Octave Pitch-Shift Data Augmentation).
    Dịch ngẫu nhiên trong dải [-max_shift_bins, +max_shift_bins] nửa cung (semitones).
    Mặc định dải ±12 semitones bao trọn 1 quãng 8 (chuyển đổi hoàn hảo giữa giọng Nam và Nữ).
    """
    def __init__(self, max_shift_bins: int = 12, prob: float = 0.8):
        self.max_shift_bins = max_shift_bins
        self.prob = prob

    def __call__(self, cqt_tensor: torch.Tensor) -> torch.Tensor:
        if random.random() > self.prob:
            return cqt_tensor
        shift = random.randint(-self.max_shift_bins, self.max_shift_bins)
        return apply_cqt_pitch_shift(cqt_tensor, shift)

### 3.3 Lop bien dang toc do va che phu pho (SpectrogramTempoAndMaskDistortion)
Co gian truc thoi gian trong khoang [0.8, 1.2] mo phong hat nhanh/cham ma giu nguyen cao do, ket hop SpecAugment che phu nhe mo phong ngat tieng hoac micro re.

In [ ]:
class SpectrogramTempoAndMaskDistortion:
    """Nonlinear time warp plus SpecAugment masks; the CQT frequency axis is never warped."""
    def __init__(self, time_scale_range=(0.5, 1.5), max_mask_t=0, max_mask_f=0, prob=0.6, knots=6):
        self.scale_range = time_scale_range
        self.max_mask_t = int(max_mask_t)
        self.max_mask_f = int(max_mask_f)
        self.prob = prob
        self.knots = knots
        if not 0 < time_scale_range[0] <= time_scale_range[1] or knots < 2:
            raise ValueError('Invalid distortion parameters')
        if self.max_mask_t < 0 or self.max_mask_f < 0:
            raise ValueError('Mask widths must be non-negative')

    def _axis(self, size, batch, device, dtype):
        increments = torch.empty(batch, 1, self.knots, device=device, dtype=dtype).uniform_(*self.scale_range)
        positions = torch.cat([torch.zeros(batch, 1, 1, device=device, dtype=dtype), increments.cumsum(-1)], -1)
        positions = 2 * positions / positions[..., -1:] - 1
        return F.interpolate(positions, size=size, mode='linear', align_corners=True)[:, 0]

    @staticmethod
    def _mask_axis(x, axis, max_width):
        if max_width <= 0 or x.shape[axis] <= 1:
            return
        width = random.randint(1, min(max_width, x.shape[axis] - 1))
        start = random.randint(0, x.shape[axis] - width)
        slices = [slice(None)] * x.ndim
        slices[axis] = slice(start, start + width)
        fill_value = x.amin(dim=(-2, -1), keepdim=True)
        x[tuple(slices)] = fill_value

    def __call__(self, spec):
        if random.random() >= self.prob:
            return spec
        was_unbatched = spec.ndim == 3
        x = spec.unsqueeze(0) if was_unbatched else spec
        b, _, h, w = x.shape
        tx = self._axis(w, b, x.device, x.dtype)
        fy = torch.linspace(-1, 1, h, device=x.device, dtype=x.dtype).expand(b, h)
        grid = torch.stack([tx[:, None, :].expand(b, h, w), fy[:, :, None].expand(b, h, w)], -1)
        out = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True).clone()
        self._mask_axis(out, axis=-1, max_width=self.max_mask_t)
        self._mask_axis(out, axis=-2, max_width=self.max_mask_f)
        return out.squeeze(0) if was_unbatched else out


### 4.1 Dinh nghia Dataset voi bo nho dem RAM (ByteHumCachedTripletDataset)
Sinh bo ba Triplet (Anchor, Positive, Negative). Cache toan bo cac chunk cua moi audio vao RAM; khi train chon ngau nhien 1 chunk moi epoch, khi validation lay co dinh chunk dau tien de dam bao tinh xac dinh.

In [ ]:
GLOBAL_CQT_RAM_CACHE: Dict[str, List[torch.Tensor]] = {}

class ByteHumCachedTripletDataset(Dataset):
    """
    Tập dữ liệu sinh bộ ba Triplet (Anchor, Positive, Negative) với cơ chế In-RAM CQT Caching:
    - Lưu danh sách tất cả các chunk CQT của mỗi file audio vào RAM, khắc phục lỗi chỉ lấy chunks[0].
    - Khi training: ngẫu nhiên chọn 1 chunk từ danh sách chunks cho mỗi epoch để học đa dạng vị trí.
    - Khi validation: lấy cố định chunks[0] để kết quả đánh giá mang tính xác định (deterministic).
    - Áp dụng Pitch-Shift (±12 semitones) và Tempo Distortion ngẫu nhiên động trên CQT.
    """
    def __init__(
        self,
        split_dir: str,
        audio_preprocessor: AudioPreprocessor,
        cqt_extractor: CQTExtractor,
        tempo_distortion: Optional[SpectrogramTempoAndMaskDistortion] = None,
        pitch_shifter: Optional[CQTPitchShiftAugmentation] = None,
        is_training: bool = True,
        song_to_idx: Optional[Dict[str, int]] = None,
        vocal_ratio: float = 0.80
    ):
        self.split_dir = split_dir
        self.audio_preprocessor = audio_preprocessor
        self.cqt_extractor = cqt_extractor
        self.tempo_distortion = tempo_distortion
        self.pitch_shifter = pitch_shifter
        self.is_training = is_training
        self.vocal_ratio = vocal_ratio

        if not split_dir or not os.path.exists(split_dir):
            raise FileNotFoundError(f"Không tìm thấy thư mục: {split_dir}")

        self.songs = sorted([s for s in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, s))])
        if song_to_idx is not None:
            self.song_to_idx = song_to_idx
        else:
            self.song_to_idx = {s: i for i, s in enumerate(self.songs)}

        self.song_data = {}
        self.humming_samples = []

        for s in self.songs:
            s_path = os.path.join(split_dir, s)
            frags = sorted([f for f in os.listdir(s_path) if os.path.isdir(os.path.join(s_path, f))])
            self.song_data[s] = {}
            for f in frags:
                f_path = os.path.join(s_path, f)
                audio_extensions = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.silent')
                audio_files = sorted(w for w in os.listdir(f_path) if w.lower().endswith(audio_extensions))
                refs = [os.path.join(f_path, w) for w in audio_files if w == 'reference.wav']
                vocals = [os.path.join(f_path, w) for w in audio_files if w.startswith('reference_vocal')]
                hums = [os.path.join(f_path, w) for w in audio_files if not w.startswith('reference')]
                if refs or vocals or hums:
                    self.song_data[s][f] = {'refs': refs, 'vocals': vocals, 'hums': hums}
                    for h in hums:
                        self.humming_samples.append((s, f, h))

    def __len__(self) -> int:
        return len(self.humming_samples)

    def _get_raw_cqts(self, path: str) -> List[torch.Tensor]:
        """Lấy danh sách tất cả các tensor CQT chunks từ RAM Cache, nếu chưa có thì trích xuất toàn bộ."""
        if path in GLOBAL_CQT_RAM_CACHE:
            return GLOBAL_CQT_RAM_CACHE[path]
        try:
            y = self.audio_preprocessor.load_audio(path)
            chunks = self.audio_preprocessor.slice_into_chunks(y)
            cqts = [self.cqt_extractor.extract(chk, variable_length=False) for chk in chunks]
        except Exception as e:
            raise RuntimeError(f"Không thể tạo CQT cho {path}") from e
        GLOBAL_CQT_RAM_CACHE[path] = cqts
        return cqts

    def _sample_chunk(self, path: str) -> torch.Tensor:
        """Chọn 1 chunk CQT: ngẫu nhiên khi train, cố định chunk 0 khi val."""
        cqts = self._get_raw_cqts(path)
        if self.is_training:
            return random.choice(cqts).clone()
        return cqts[0].clone()

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        song_a, frag_a, hum_a = self.humming_samples[idx]
        
        # 1. Anchor: Tiếng ngân nga (Domain 1.0)
        cqt_a = self._sample_chunk(hum_a)
        if self.is_training:
            if self.tempo_distortion: cqt_a = self.tempo_distortion(cqt_a)
            if self.pitch_shifter: cqt_a = self.pitch_shifter(cqt_a)

        # 2. Positive: Bản thu gốc hoặc vocal tách rời
        pool = self.song_data[song_a][frag_a]
        if self.is_training and pool['vocals'] and random.random() < self.vocal_ratio:
            pos_path, domain_p = random.choice(pool['vocals']), 0.0
        elif pool['refs']:
            pos_path, domain_p = (random.choice(pool['refs']) if self.is_training else pool['refs'][0]), 0.0
        elif pool['vocals']:
            pos_path, domain_p = (random.choice(pool['vocals']) if self.is_training else pool['vocals'][0]), 0.0
        else:
            cands = sorted(h for h in pool['hums'] if h != hum_a)
            if not cands:
                raise RuntimeError(f"Thiếu positive phù hợp cho {song_a}/{frag_a}")
            pos_path, domain_p = (random.choice(cands) if self.is_training else cands[0]), 1.0
            
        cqt_p = self._sample_chunk(pos_path)
        if self.is_training and domain_p == 1.0:
            if self.pitch_shifter: cqt_p = self.pitch_shifter(cqt_p)

        # 3. Negative: Bản thu của bài hát khác hoàn toàn
        neg_song_candidates = [s for s in self.songs if s != song_a and self.song_data[s]]
        if not neg_song_candidates:
            raise RuntimeError(f"Không có negative hợp lệ cho {song_a}")
        if self.is_training:
            neg_song = random.choice(neg_song_candidates)
            neg_frag = random.choice(sorted(self.song_data[neg_song]))
        else:
            neg_song = neg_song_candidates[idx % len(neg_song_candidates)]
            neg_fragments = sorted(self.song_data[neg_song])
            neg_frag = neg_fragments[idx % len(neg_fragments)]
        neg_pool = self.song_data[neg_song][neg_frag]
        if self.is_training:
            if random.random() > 0.5 and neg_pool['refs']:
                neg_path, domain_n = random.choice(neg_pool['refs']), 0.0
            elif neg_pool['hums']:
                neg_path, domain_n = random.choice(neg_pool['hums']), 1.0
            elif neg_pool['vocals']:
                neg_path, domain_n = random.choice(neg_pool['vocals']), 0.0
            else:
                raise RuntimeError(f"Thiếu negative audio cho {neg_song}/{neg_frag}")
        elif neg_pool['refs']:
            neg_path, domain_n = neg_pool['refs'][0], 0.0
        elif neg_pool['hums']:
            neg_path, domain_n = neg_pool['hums'][0], 1.0
        elif neg_pool['vocals']:
            neg_path, domain_n = neg_pool['vocals'][0], 0.0
        else:
            raise RuntimeError(f"Thiếu negative audio cho {neg_song}/{neg_frag}")
            
        cqt_n = self._sample_chunk(neg_path)

        song_label = self.song_to_idx.get(song_a, 0)
        neg_song_label = self.song_to_idx.get(neg_song, 0)

        return {
            'anchor': cqt_a,
            'positive': cqt_p,
            'negative': cqt_n,
            'song_idx': torch.tensor(song_label, dtype=torch.long),
            'negative_song_idx': torch.tensor(neg_song_label, dtype=torch.long),
            'domain_anchor': torch.tensor([1.0], dtype=torch.float),
            'domain_positive': torch.tensor([domain_p], dtype=torch.float),
            'domain_negative': torch.tensor([domain_n], dtype=torch.float)
        }

### 5.1 Dinh nghia tang dao nguoc dao ham (Gradient Reversal Layer - GRL)
Dao nguoc chieu gradient (nhan voi -lambda) khi truyen nguoc tu bo phan loai mien sang feature extractor, giup mo hinh hoc dac trung bat bien mien am thanh (Domain Invariant Features) giua am thanh phong thu va tieng ngan nga.

In [ ]:
from torch.autograd import Function

class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_val: float):
        ctx.lambda_val = lambda_val
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    """Lớp đảo ngược gradient với hệ số lambda điều chỉnh động theo epoch."""
    def __init__(self, lambda_val: float = 0.2):
        super(GradientReversalLayer, self).__init__()
        self.lambda_val = lambda_val

    def set_lambda(self, lambda_val: float):
        self.lambda_val = lambda_val

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return GradientReversalFunction.apply(x, self.lambda_val)

### 5.2 Dinh nghia bo phan loai mien cuc bo (LocalDomainClassifier)
Phan loai mien am thanh truc tiep tren chuoi dac trung cuc bo [B, T=20, 128] bang kien truc Conv1D da tang va LayerNorm. Khac phuc han che cua phien ban cu von chi nhan vector embedding toan cuc 128-D.

In [ ]:
class LocalDomainClassifier(nn.Module):
    """
    Bộ phân loại miền cục bộ (Local Domain Classifier) theo kiến trúc ByteHum:
    - Nhận đầu vào: Chuỗi đặc trưng cục bộ x [B, T, 128] từ Conformer encoder.
    - Áp dụng các tầng tích chập 1D theo trục thời gian để học đặc trưng miền cục bộ.
    - Dự đoán phân phối nhãn miền (0: Vocal phong thu, 1: Tiếng humming) trên từng phân đoạn thời gian.
    """
    def __init__(self, feature_dim: int = 128, hidden_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(feature_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GroupNorm(1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, 128] -> transpose thành [B, 128, T] cho Conv1D
        x_t = x.transpose(1, 2)
        feat = self.conv(x_t) # [B, hidden_dim, T]
        # Frame-level logits: [B, T]
        frame_logits = self.fc(feat.transpose(1, 2)).squeeze(-1)
        # Tổng hợp logits qua trục thời gian T
        return frame_logits.mean(dim=-1) # [B]

### 5.3 Ham nap mo hinh nen tang CoverHunter Conformer tu checkpoint Stage 1
Nap tron ven 265/265 tensor tu checkpoint SHS100K pre-trained (6 khoi Conformer, Conv2d Subsampling 4x, Attention Pooling 128-D).

In [ ]:
def build_coverhunter_model(ckpt_path: str = None, device: str = "cpu", allow_random_init: bool = False):
    """
    Khởi tạo mô hình CoverHunter Conformer chính thức (9.91M tham số, 6 khối Conformer, Attention Pooling 128-D).
    Nạp trọn vẹn 265/265 tensor từ checkpoint SHS100K pre-trained.
    """
    # Cấu hình siêu tham số mặc định chuẩn xác của CoverHunter SHS100K
    hp = {
        'input_dim': 96, 'embed_dim': 128,
        'encoder': {'output_dims': 128, 'num_blocks': 6, 'attention_dim': 256},
        'pool_type': 'attention',
        'ce': {'output_dims': 30000, 'weight': 1.0, 'gamma': 2},
        'triplet': {'margin': 0.3, 'weight': 0.1},
        'center': {'weight': 0.0}
    }

    ch_model = Model(hp)

    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state_dict = ckpt.get("generator", ckpt.get("model_state_dict", ckpt))
        ch_model.load_state_dict(state_dict)
        total_p = sum(p.numel() for p in ch_model.parameters())
        print(f"[+] Nạp thành công 265/265 tensors ({total_p:,} params) từ CoverHunter Pretrained:\n    -> {ckpt_path}")
    elif allow_random_init:
        print("[*] Khoi tao backbone tam thoi; checkpoint fine-tuned se duoc nap ngay sau do.")
    else:
        raise FileNotFoundError(f"Bat buoc phai co checkpoint Stage 1 SHS100K: {ckpt_path}")

    ch_model = ch_model.to(device)
    return ch_model

### 5.4 Dinh nghia mo hinh tich hop Conformer-ByteHum voi Local Features va LAC
Ket hop Conformer backbone voi bo phan loai can chinh cuc bo Local Alignment Classifier (LAC) qua phep toan True MaxMean va bo phan loai mien cuc bo qua GRL. Ham forward tra ve dong thoi Global Embedding [B, 128] va chuoi Local Features [B, T=20, 128].

In [ ]:
class LocalAlignmentClassifier(nn.Module):
    """
    Local Alignment Classifier (LAC) theo kiến trúc ByteHum (ICASSP 2024):
    - Trọng số Prototype: W in R^{K x L x C}, trong đó K là số lớp bài hát (train songs),
      L là số phân đoạn prototype cục bộ (mặc định L=4), C=128.
    - Điểm tương đồng MaxMean: S(X, W_k) = (1/L) * sum_{l=1}^L max_{t=1}^T cos(x_t, w_{k,l}).
    - Cho phép căn chỉnh cục bộ bất biến thời gian và trôi dạt nhịp điệu giữa Humming và Vocal.
    """
    def __init__(self, num_classes: int = 210, feature_dim: int = 128, num_prototypes: int = 4):
        super().__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.num_prototypes = num_prototypes
        # Khởi tạo ma trận prototype W [K, L, C]
        self.prototypes = nn.Parameter(torch.randn(num_classes, num_prototypes, feature_dim) * 0.05)
        self.temperature = nn.Parameter(torch.tensor(10.0))

    def forward(self, local_feat: torch.Tensor) -> torch.Tensor:
        # local_feat: [B, T, 128]
        # Chuẩn hóa L2 trên trục đặc trưng 128-D
        x_norm = F.normalize(local_feat, p=2, dim=-1) # [B, T, C]
        w_norm = F.normalize(self.prototypes, p=2, dim=-1) # [K, L, C]

        # Tính cosine similarity giữa toàn bộ T frames và K*L prototypes qua phép nhân ma trận:
        w_flat = w_norm.view(-1, self.feature_dim).t() # [C, K*L]
        sim = torch.matmul(x_norm, w_flat) # [B, T, K*L]
        B, T, _ = sim.shape
        sim = sim.view(B, T, self.num_classes, self.num_prototypes) # [B, T, K, L]

        # Phep toan MaxMean: Max theo truc thoi gian T, Mean theo so prototypes L
        max_t, _ = sim.max(dim=1) # [B, K, L]
        logits = max_t.mean(dim=-1) # [B, K]
        return logits * self.temperature.clamp(min=1.0, max=100.0)

class ConformerByteHumModel(nn.Module):
    """
    MÔ HÌNH HOÀN CHỈNH CONFORMER-BYTEHUM CHO QBH DOMAIN ADAPTATION:
    - Backbone: Conformer (6 khối, Conv2d Subsampling 4x -> T_enc=20 frames cho 10s audio)
    - Output 1: global_emb [B, 128] (qua Attentive Statistics Pooling + L2 Normalize)
    - Output 2: local_feat [B, T=20, 128] (chuỗi đặc trưng cục bộ trước pooling)
    - Output 3: lac_logits [B, K] (Local Alignment Classifier phân loại bài hát qua MaxMean)
    - Output 4: domain_logits [B] (Local Domain Classifier phân biệt miền Vocal vs Humming)
    """
    def __init__(self, ckpt_path: str = None, input_dim: int = 96, embedding_dim: int = 128, num_classes: int = 210, lambda_grl: float = 0.2, allow_random_init: bool = False):
        super().__init__()
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        if embedding_dim not in (128, 256, 512):
            raise ValueError('embedding_dim must be 128, 256 or 512')
        self.local_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)
        self.global_projection = nn.Identity() if embedding_dim == 128 else nn.Linear(128, embedding_dim)

        # Nạp backbone Conformer pre-trained từ CoverHunter Stage 1
        self.feature_extractor = build_coverhunter_model(ckpt_path=ckpt_path, allow_random_init=allow_random_init)
        
        # Đóng băng các tầng phụ không dùng của Stage 1 (ce_layer, center_loss)
        if hasattr(self.feature_extractor, '_ce_layer'):
            for p in self.feature_extractor._ce_layer.parameters(): p.requires_grad = False
        if hasattr(self.feature_extractor, '_center_loss'):
            for p in self.feature_extractor._center_loss.parameters(): p.requires_grad = False

        self.grl = GradientReversalLayer(lambda_val=lambda_grl)
        self.domain_classifier = LocalDomainClassifier(feature_dim=embedding_dim, hidden_dim=64)
        self.lac = LocalAlignmentClassifier(num_classes=num_classes, feature_dim=embedding_dim, num_prototypes=4)

    def train(self, mode: bool = True):
        super().train(mode)
        # Preserve the SHS100K CMVN statistics while fine-tuning on humming audio.
        if mode and hasattr(self.feature_extractor, '_global_cmvn'):
            self.feature_extractor._global_cmvn.eval()
        return self

    def set_lambda(self, lambda_val: float):
        self.grl.set_lambda(lambda_val)

    def _prepare_input(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 4: x = x.squeeze(1)
        if x.dim() == 3:
            if x.shape[1] == 96 and x.shape[2] != 96: x = x.transpose(1, 2)
        elif x.dim() == 2:
            if x.shape[0] == 96 and x.shape[1] != 96: x = x.t().unsqueeze(0)
            else: x = x.unsqueeze(0)
        return x

    def extract_backbone_features(self, x: torch.Tensor) -> torch.Tensor:
        """Trích xuất chuỗi đặc trưng cục bộ [B, T=20, 128] từ Conformer encoder trước ASP pooling."""
        x_prep = self._prepare_input(x)
        x_cmvn = self.feature_extractor._global_cmvn(x_prep.transpose(1, 2)).transpose(1, 2)
        xs_lens = torch.full([x_cmvn.size(0)], fill_value=x_cmvn.size(1), dtype=torch.long, device=x_cmvn.device)
        x_enc, _ = self.feature_extractor._encoder(x_cmvn, xs_lens=xs_lens, decoding_chunk_size=-1)
        return x_enc # [B, T=20, 128]

    def extract_local_features(self, x):
        return self.local_projection(self.extract_backbone_features(x))

    def extract_global_embedding(self, local_feat: torch.Tensor) -> torch.Tensor:
        """Chiếu chuỗi đặc trưng cục bộ qua Attentive Statistics Pooling thành Global 128-D Embedding."""
        emb = self.feature_extractor._pool_layer(local_feat)
        return F.normalize(self.global_projection(emb), p=2, dim=-1)

    def extract_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Hàm tiện ích trích xuất trực tiếp Global Embedding 128-D từ phổ CQT đầu vào."""
        return self.extract_global_embedding(self.extract_backbone_features(x))

    def forward(self, x: torch.Tensor, compute_lac: bool = True, compute_domain: bool = True):
        """Return global/local features and only the auxiliary heads requested by the run."""
        backbone_feat = self.extract_backbone_features(x)
        local_feat = self.local_projection(backbone_feat)
        global_emb = self.extract_global_embedding(backbone_feat)
        lac_logits = self.lac(local_feat) if compute_lac else None
        domain_logits = self.domain_classifier(self.grl(local_feat)) if compute_domain else None
        return global_emb, local_feat, lac_logits, domain_logits

    def freeze_lower_layers(self, num_blocks_to_freeze: int = 3, freeze_frontend: bool = True):
        """Đóng băng tầng frontend và các khối Conformer đầu tiên theo chiến lược Deep Freeze."""
        if freeze_frontend:
            if hasattr(self.feature_extractor, '_global_cmvn'):
                for p in self.feature_extractor._global_cmvn.parameters(): p.requires_grad = False
            if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'embed'):
                for p in self.feature_extractor._encoder.embed.parameters(): p.requires_grad = False

        if hasattr(self.feature_extractor, '_encoder') and hasattr(self.feature_extractor._encoder, 'encoders'):
            for i, block in enumerate(self.feature_extractor._encoder.encoders):
                if i < num_blocks_to_freeze:
                    for p in block.parameters(): p.requires_grad = False
                else:
                    for p in block.parameters(): p.requires_grad = True

        if hasattr(self.feature_extractor, '_pool_layer'):
            for p in self.feature_extractor._pool_layer.parameters(): p.requires_grad = True
        for p in self.domain_classifier.parameters(): p.requires_grad = True
        for p in self.lac.parameters(): p.requires_grad = True

### 6.1 Nap checkpoint fine-tuned va khoi tao test dataset


In [ ]:
# Nap checkpoint fine-tuned va khoi tao dataset test doc lap voi notebook training.
def find_finetuned_checkpoint(filename="conformer_bytehum_full_d128_timegrid_lat_v4_best.pth"):
    # 1. Bien moi truong chi dinh truc tiep
    env_ckpt = os.environ.get('QBH_CHECKPOINT')
    if env_ckpt and os.path.exists(env_ckpt):
        return os.path.abspath(env_ckpt)
        
    # 2. Cac thu muc uu tien cuc bo & Kaggle working
    candidates = [
        os.path.join(CHECKPOINT_DIR, filename),
        os.path.join("..", "checkpoints", filename),
        os.path.join("checkpoints", filename),
        os.path.join(os.path.abspath(".."), "02_stage2_humming_qbh", "checkpoints", filename),
        os.path.join("/kaggle/working/checkpoints", filename),
        os.path.join("/kaggle/working", filename)
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
            
    # 3. Quet tu dong trong Kaggle Input (khi attach checkpoint tu notebook train sang danh gia)
    if IS_KAGGLE and os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(".pth") and any(k in f.lower() for k in ["timegrid_lat_v4", "lat_v4_best"]):
                    return os.path.join(root, f)

    # 4. Quet de quy trong thu muc du an hien tai
    base_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
    for root, _, files in os.walk(base_dir):
        if filename in files:
            return os.path.join(root, filename)
            
    return os.path.join(CHECKPOINT_DIR, filename)

finetuned_ckpt_path = find_finetuned_checkpoint("conformer_bytehum_full_d128_timegrid_lat_v4_best.pth")

if not os.path.exists(finetuned_ckpt_path):
    raise FileNotFoundError(
        f"Khong tim thay checkpoint fine-tuned: {finetuned_ckpt_path}. "
        "Hay chay notebook training truoc hoac attach checkpoint vao Kaggle."
    )

checkpoint_metadata = torch.load(finetuned_ckpt_path, map_location="cpu", weights_only=False)
checkpoint_state = checkpoint_metadata.get('model_state_dict', checkpoint_metadata)
prototype_key = 'lac.prototypes'
if prototype_key not in checkpoint_state:
    raise KeyError(f"Checkpoint thieu tensor bat buoc: {prototype_key}")
EMBEDDING_DIM = int(checkpoint_state[prototype_key].shape[-1])
NUM_TRAIN_SONGS = int(checkpoint_state[prototype_key].shape[0])

epoch_info = checkpoint_metadata.get('epoch', 'N/A')
best_mrr_info = checkpoint_metadata.get('best_val_mrr', 'N/A')
run_name_info = checkpoint_metadata.get('run_name', 'N/A')

test_dataset = ByteHumCachedTripletDataset(
    split_dir=TEST_DIR,
    audio_preprocessor=prep,
    cqt_extractor=cqt_ext,
    tempo_distortion=None,
    pitch_shifter=None,
    is_training=False,
    vocal_ratio=0.0
)
print(f"[+] Dinh vi thanh cong Checkpoint Fine-Tuned (v4):\n    -> {finetuned_ckpt_path}")
print(f"    - Thong tin huan luyen   : Run '{run_name_info}' | Epoch {epoch_info} | Best Val MRR: {best_mrr_info:.4f}" if isinstance(best_mrr_info, float) else f"    - Thong tin huan luyen   : Run '{run_name_info}' | Epoch {epoch_info} | Best Val MRR: {best_mrr_info}")
print(f"    - Kich thuoc Vector (Dim): {EMBEDDING_DIM} | So lop train: {NUM_TRAIN_SONGS}")
print(f"    - Test queries trong tap : {len(test_dataset.humming_samples):,} queries")


### 7.1 Chuan bi danh sach bai hat Gallery va tap cau hoi kiem thu (Test Queries)
Quet toan bo danh sach cac bai hat goc trong kho nhac Gallery (music_database) va toan bo tap humming kiem thu thuc te. Phan chia ro rang cac tap con: All Test Queries, Seen Song Queries, va Unseen Song Queries de chong danh gia sai lech.

In [ ]:
TOP_K_LIST = [1, 5, 10, 20]

all_gallery_songs = sorted([
    os.path.splitext(f)[0] for f in os.listdir(MUSIC_DB_DIR)
    if f.lower().endswith(('.mp3', '.wav'))
]) if (MUSIC_DB_DIR and os.path.exists(MUSIC_DB_DIR)) else []
print(f"[+] Tổng số bài hát trong Gallery Kho Nhạc Đối Soát: {len(all_gallery_songs)} bài")

all_test_queries = []
if TEST_DIR and os.path.exists(TEST_DIR):
    for root, dirs, files in os.walk(TEST_DIR):
        for f in sorted(files):
            if f.lower().endswith(('.wav', '.mp3')) and not f.startswith('reference'):
                rel = os.path.relpath(root, TEST_DIR)
                gt_song = rel.split(os.sep)[0]
                if gt_song in all_gallery_songs:
                    all_test_queries.append((os.path.join(root, f), gt_song))

# Phân tách tập kiểm thử thành Seen Song (đã có trong tập Train) và Unseen Song (chưa từng thấy)
train_songs_set = {s for s in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, s))} if TRAIN_DIR and os.path.exists(TRAIN_DIR) else set()
seen_song_queries = [(q, s) for q, s in all_test_queries if s in train_songs_set]
unseen_song_queries = [(q, s) for q, s in all_test_queries if s not in train_songs_set]

print(f"[+] Thống kê tập câu hỏi kiểm thử humming (Test Queries):")
print(f"    - Toàn bộ tập kiểm thử (All Queries)    : {len(all_test_queries):,} queries")
print(f"    - Tập Seen Song (Bài hát có trong train): {len(seen_song_queries):,} queries")
print(f"    - Tập Unseen Song (Bài hát hoàn toàn mới): {len(unseen_song_queries):,} queries")

### 7.2 Dinh nghia ham danh gia da doan True MaxMean (evaluate_cross_domain_retrieval)
Trich xuat va cache san chi muc embedding cho toan bo kho nhac Gallery vao bo nho RAM. Ap dung thuat toan True MaxMean chuan xac cho tung cau query, tinh toan day du cac chi so: Top-1, Top-5, Top-10, Top-20 Recall, MRR va duong cong CMC.

In [ ]:
def build_retrieval_index(matrix, kind='flat', nlist=64, nprobe=16):
    matrix = np.ascontiguousarray(matrix, dtype=np.float32)
    if matrix.ndim != 2 or not len(matrix) or not np.isfinite(matrix).all():
        raise ValueError('Invalid index vectors')
    faiss.normalize_L2(matrix)
    d = matrix.shape[1]
    if kind == 'flat':
        index = faiss.IndexFlatIP(d)
    elif kind == 'hnsw':
        index = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efConstruction = 200
        index.hnsw.efSearch = 128
    elif kind == 'ivf':
        if len(matrix) < 39 * nlist:
            raise ValueError('IVF requires at least 39*nlist training vectors; reduce nlist or use flat/HNSW')
        index = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, nlist, faiss.METRIC_INNER_PRODUCT)
        index.train(matrix)
        index.nprobe = min(nprobe,nlist)
    else:
        raise ValueError(kind)
    index.add(matrix)
    return index

@torch.no_grad()
def encode_full_audio(model, waveform, batch_size=16):
    model.eval()
    global_parts, local_parts = [], []
    chunks = prep.slice_into_chunks(waveform)
    for start in range(0,len(chunks),batch_size):
        batch = torch.stack([cqt_ext.extract(c) for c in chunks[start:start+batch_size]]).to(device)
        features = model.extract_backbone_features(batch)
        global_parts.append(model.extract_global_embedding(features).cpu().numpy())
        local = F.normalize(model.local_projection(features),dim=-1)
        local_parts.append(local.reshape(-1,local.shape[-1]).cpu().numpy())
    return np.concatenate(global_parts), np.concatenate(local_parts)



In [ ]:
from tqdm import tqdm

def evaluate_cross_domain_retrieval(
    eval_model,
    gallery_songs,
    queries,
    model_name="Model",
    chunk_sec=10.0,
    hop_sec=3.0,
    gallery_max_sec=None,
    precomputed_gallery=None,
    max_eval_queries=None
):
    """
    Đánh giá độ chính xác truy vấn đa đoạn (Multi-Chunk True MaxMean):
    - Hỗ trợ tái sử dụng precomputed_gallery để tăng tốc khi đánh giá nhiều mô hình.
    - True MaxMean: S(Q, R) = (1/|Q|) * sum_{i} max_{j} cos(q_i, r_j).
    - Xuất đầy đủ các chỉ số: Top-1, Top-5, Top-10, Top-20, MRR, Mean Rank và đường cong CMC.
    """
    eval_model.eval()
    chunk_samples = int(chunk_sec * 16000)
    hop_samples = int(hop_sec * 16000)
    
    # 1. Trích xuất hoặc tái sử dụng chỉ mục kho nhạc Gallery
    if precomputed_gallery is not None:
        gallery_matrix, chunk_song_map, valid_gallery = precomputed_gallery
        print(f"    [+] Tái sử dụng Gallery cache: {len(valid_gallery)} bài ({len(gallery_matrix):,} chunk vectors 128-D)")
    else:
        print(f"Indexing {len(gallery_songs)} songs; duration limit={gallery_max_sec}; model={model_name}")
        all_chunk_embs = []
        chunk_song_indices = []
        valid_gallery = []
        
        with torch.no_grad():
            for s in tqdm(gallery_songs, desc=f"Indexing Gallery [{model_name}]", leave=False):
                vocal_p = os.path.join(VOCAL_DIR, f"{s}_vocal.wav") if VOCAL_DIR else ""
                mp3_p = os.path.join(MUSIC_DB_DIR, f"{s}.mp3") if MUSIC_DB_DIR else ""
                target_p = vocal_p if os.path.exists(vocal_p) else mp3_p
                if not os.path.exists(target_p):
                    continue

                try:
                    y_s, sr_s = sf.read(target_p, always_2d=False, dtype='float32')
                    if len(y_s.shape) > 1: y_s = np.mean(y_s, axis=1)
                    if sr_s != 16000: y_s = librosa.resample(y_s, orig_sr=sr_s, target_sr=16000)
                except Exception:
                    try:
                        y_s, _ = librosa.load(target_p, sr=16000, duration=gallery_max_sec)
                    except Exception:
                        continue
                        
                y_s = y_s if gallery_max_sec is None else y_s[:int(16000 * gallery_max_sec)]
                chunks = [y_s[st : st + chunk_samples] for st in range(0, len(y_s) - chunk_samples + 1, hop_samples)]
                if not chunks: chunks = [y_s[:chunk_samples]]
                    
                _, embs = encode_full_audio(eval_model,y_s)
                s_idx = len(valid_gallery)
                valid_gallery.append(s)
                for emb in embs:
                    all_chunk_embs.append(emb)
                    chunk_song_indices.append(s_idx)
                    
        gallery_matrix = np.array(all_chunk_embs, dtype=np.float32)
        chunk_song_map = np.array(chunk_song_indices, dtype=np.int32)
        print(f"    [+] Hoàn tất Gallery: {len(valid_gallery)} bài ({len(gallery_matrix):,} chunk vectors 128-D)")

    song_chunk_masks = [np.where(chunk_song_map == sid)[0] for sid in range(len(valid_gallery))]

    # 2. Đánh giá tập truy vấn (Queries Evaluation)
    eval_queries = queries if (max_eval_queries is None or max_eval_queries <= 0) else queries[:max_eval_queries]
    
    hits = {k: 0 for k in TOP_K_LIST}
    cmc_ranks = np.zeros(20, dtype=float)
    mrrs = []
    successful_queries = 0
    failed_queries = 0
    
    pbar = tqdm(eval_queries, desc=f"Evaluating Queries [{model_name}]", leave=False)
    with torch.no_grad():
        for q_path, gt_song in pbar:
            if gt_song not in valid_gallery:
                failed_queries += 1
                continue
            gt_idx = valid_gallery.index(gt_song)
            try:
                y_q, sr_q = sf.read(q_path, always_2d=False, dtype='float32')
                if len(y_q.shape) > 1: y_q = np.mean(y_q, axis=1)
                if sr_q != 16000: y_q = librosa.resample(y_q, orig_sr=sr_q, target_sr=16000)
                if len(y_q) < chunk_samples:
                    y_q = np.pad(y_q, (0, chunk_samples - len(y_q)), mode='constant')
                y_q = y_q[:16000 * 15]
                
                q_chunks = [y_q[st : st + chunk_samples] for st in range(0, len(y_q) - chunk_samples + 1, hop_samples)]
                if not q_chunks: q_chunks = [y_q[:chunk_samples]]
                
                q_tensors = torch.stack([cqt_ext.extract(c, variable_length=False) for c in q_chunks], dim=0).to(device)
                q_embs = F.normalize(eval_model.extract_local_features(q_tensors), p=2, dim=-1)
                q_embs = q_embs.reshape(-1, q_embs.shape[-1]).cpu().numpy()
            except Exception:
                failed_queries += 1
                continue
                
            sim_mat = np.dot(q_embs, gallery_matrix.T)
            
            # Tính điểm True MaxMean cho từng bài hát trong Gallery
            song_scores = np.zeros(len(valid_gallery), dtype=np.float32)
            for sid, c_indices in enumerate(song_chunk_masks):
                if len(c_indices) > 0:
                    sub_sims = sim_mat[:, c_indices]
                    max_per_query = np.max(sub_sims, axis=1)
                    song_scores[sid] = np.mean(max_per_query)
                else:
                    song_scores[sid] = -1.0
                    
            ranked_indices = np.argsort(-song_scores)
            
            try:
                rank = int(np.where(ranked_indices == gt_idx)[0][0]) + 1
            except IndexError:
                rank = 999999
                
            mrrs.append(1.0 / rank)
            if rank <= 20: cmc_ranks[rank - 1] += 1
            for k in TOP_K_LIST:
                if rank <= k: hits[k] += 1
            successful_queries += 1
                    
    total_eval = max(1, len(mrrs))
    cmc_curve = np.cumsum(cmc_ranks) / total_eval * 100.0
    
    results = {
        'top1': hits[1] / total_eval * 100.0,
        'top5': hits[5] / total_eval * 100.0,
        'top10': hits[10] / total_eval * 100.0,
        'top20': hits[20] / total_eval * 100.0,
        'mrr': float(np.mean(mrrs)) if mrrs else 0.0,
        'cmc': cmc_curve,
        'total_evaluated': total_eval,
        'successful': successful_queries,
        'failed': failed_queries
    }
    return results, (gallery_matrix, chunk_song_map, valid_gallery)

### 7.3 Danh gia hieu qua Zero-Shot (Mo hinh truoc khi tinh chinh)
Kiem thu nang luc nhan dien cua trong so goc Stage 1 CoverHunter khi chua qua thich ung mien de lam moc doi chung chuan xac.

In [ ]:
print("\n[*] Khởi chạy đánh giá đa đoạn (Multi-Chunk True MaxMean) cho [Zero-Shot Model]...")
zero_shot_model = ConformerByteHumModel(ckpt_path=STAGE1_CKPT, input_dim=96, embedding_dim=128, num_classes=NUM_TRAIN_SONGS, lambda_grl=0.2).to(device)

res_zs, zs_gallery_cache = evaluate_cross_domain_retrieval(
    zero_shot_model,
    all_gallery_songs,
    all_test_queries,
    model_name="Zero-Shot Conformer"
)

print(f"[+] Kết quả Zero-Shot: Top-1 = {res_zs['top1']:.2f}% | Top-5 = {res_zs['top5']:.2f}% | Top-10 = {res_zs['top10']:.2f}% | MRR = {res_zs['mrr']:.4f}")

### 7.4 Danh gia hieu qua Fine-Tuned (Mo hinh sau khi thich ung mien)
Nap checkpoint toi uu nhat da huan luyen va danh gia do chinh xac truy van da doan tren toan bo tap kiem thu.

In [ ]:
print("\n[*] Khởi chạy đánh giá đa đoạn (Multi-Chunk True MaxMean) cho [Fine-Tuned Model]...")
finetuned_eval_model = ConformerByteHumModel(ckpt_path=None, input_dim=96, embedding_dim=EMBEDDING_DIM, num_classes=NUM_TRAIN_SONGS, lambda_grl=0.2, allow_random_init=True).to(device)

if os.path.exists(finetuned_ckpt_path):
    ckpt_ft = torch.load(finetuned_ckpt_path, map_location=device, weights_only=False)
    finetuned_eval_model.load_state_dict(ckpt_ft['model_state_dict'])
    print(f"    [+] Nạp thành công trọng số tinh chỉnh tối ưu từ: {finetuned_ckpt_path}")
else:
    raise FileNotFoundError(f'Khong tim thay checkpoint fine-tuned: {finetuned_ckpt_path}')

res_ft, ft_gallery_cache = evaluate_cross_domain_retrieval(
    finetuned_eval_model,
    all_gallery_songs,
    all_test_queries,
    model_name="Fine-Tuned Conformer"
)

print(f"[+] Kết quả Fine-Tuned: Top-1 = {res_ft['top1']:.2f}% | Top-5 = {res_ft['top5']:.2f}% | Top-10 = {res_ft['top10']:.2f}% | MRR = {res_ft['mrr']:.4f}")

### 7.5 Bang so sanh ket qua dinh luong giua Zero-Shot va Fine-Tuned
In bang so sanh chi tiet cac chi so Hit@1, Hit@5, Hit@10, Hit@20 va MRR giua mo hinh truoc va sau khi duoc thich ung mien kem theo muc cai thien (Delta).

In [ ]:
delta_top1 = res_ft['top1'] - res_zs['top1']
delta_top5 = res_ft['top5'] - res_zs['top5']
delta_top10 = res_ft['top10'] - res_zs['top10']
delta_top20 = res_ft['top20'] - res_zs['top20']
delta_mrr = res_ft['mrr'] - res_zs['mrr']

print("\n" + "=" * 98)
print(f"{'MÔ HÌNH ĐỐI SOÁT':^35} | {'Hit@1 (%)':^12} | {'Hit@5 (%)':^12} | {'Hit@10 (%)':^12} | {'Hit@20 (%)':^12} | {'MRR':^10}")
print("=" * 98)
print(f"{'Zero-Shot Conformer (Pre-trained)':<35} | {res_zs['top1']:^12.2f} | {res_zs['top5']:^12.2f} | {res_zs['top10']:^12.2f} | {res_zs['top20']:^12.2f} | {res_zs['mrr']:^10.4f}")
print(f"{'Fine-Tuned Conformer (Domain Adapt)':<35} | {res_ft['top1']:^12.2f} | {res_ft['top5']:^12.2f} | {res_ft['top10']:^12.2f} | {res_ft['top20']:^12.2f} | {res_ft['mrr']:^10.4f}")
print("-" * 98)
print(f"{'Mức Cải Thiện (Δ = FT - ZS)':<35} | {f'+{delta_top1:.2f}' if delta_top1>=0 else f'{delta_top1:.2f}':^12} | {f'+{delta_top5:.2f}' if delta_top5>=0 else f'{delta_top5:.2f}':^12} | {f'+{delta_top10:.2f}' if delta_top10>=0 else f'{delta_top10:.2f}':^12} | {f'+{delta_top20:.2f}' if delta_top20>=0 else f'{delta_top20:.2f}':^12} | {f'+{delta_mrr:.4f}' if delta_mrr>=0 else f'{delta_mrr:.4f}':^10}")
print("=" * 98)
print(f"[*] Thống kê: Đánh giá thành công {res_ft['successful']:,}/{res_ft['total_evaluated']:,} queries ({res_ft['failed']} queries thất bại).")

### 7.6 Ve duong cong nhan dien tich luy CMC (Cumulative Match Characteristic)
Truc quan hoa ty le nhan dien thanh cong khi mo rong thu hang Rank tu 1 den 20 giua Zero-Shot va Fine-Tuned Conformer.

In [ ]:
ranks = np.arange(1, 21)
cmc_zeroshot = res_zs['cmc']
cmc_finetuned = res_ft['cmc']

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=120)
ax.plot(ranks, cmc_zeroshot, marker='s', linestyle='-.', color='#3498db', linewidth=2.0, label=f'Zero-Shot Conformer (Top-1: {res_zs["top1"]:.2f}%, MRR: {res_zs["mrr"]:.4f})')
ax.plot(ranks, cmc_finetuned, marker='o', linestyle='-', color='#27ae60', linewidth=2.5, label=f'Fine-Tuned Conformer (Top-1: {res_ft["top1"]:.2f}%, MRR: {res_ft["mrr"]:.4f})')

ax.set_title('Đường Cong Nhận Diện Tích Lũy CMC (Rank 1 -> 20)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Thứ Hạng Đánh Giá (Rank K)', fontsize=10)
ax.set_ylabel('Tỷ Lệ Nhận Diện Chính Xác (%)', fontsize=10)
ax.set_xticks(ranks)
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(frameon=True, fontsize=10)
plt.tight_layout()
cmc_chart_path = os.path.join(OUTPUT_DIR, "cmc_curve.png")
plt.savefig(cmc_chart_path, dpi=150)
plt.show()
print(f"[+] Đã lưu biểu đồ CMC Curve tại: {cmc_chart_path}")

### 8.1 Thuat toan Subsequence DTW co tu dong can chinh cao do (compute_cqt_dtw_distance)
Su dung Subsequence DTW open-begin/open-end tren CQT do dai dong (variable-length), de query humming co the khop voi bat ky doan nao trong reference. Pho trung binh duoc dung de chon nhanh do lech tone tu -12 den +12 semitones truoc khi tinh DTW.

In [ ]:
try:
    from numba import njit
except ImportError:
    def njit(fn):
        return fn


@njit
def _subsequence_dtw_distance(cost):
    """Open-begin/open-end DTW: the whole query must match a reference subsequence."""
    query_frames, reference_frames = cost.shape
    previous = np.zeros(reference_frames + 1, dtype=np.float32)
    previous_lengths = np.zeros(reference_frames + 1, dtype=np.int32)

    for query_idx in range(query_frames):
        current = np.full(reference_frames + 1, np.inf, dtype=np.float32)
        current_lengths = np.zeros(reference_frames + 1, dtype=np.int32)
        for reference_idx in range(1, reference_frames + 1):
            best_parent = previous[reference_idx - 1]
            best_length = previous_lengths[reference_idx - 1]
            if previous[reference_idx] < best_parent:
                best_parent = previous[reference_idx]
                best_length = previous_lengths[reference_idx]
            if current[reference_idx - 1] < best_parent:
                best_parent = current[reference_idx - 1]
                best_length = current_lengths[reference_idx - 1]
            current[reference_idx] = cost[query_idx, reference_idx - 1] + best_parent
            current_lengths[reference_idx] = best_length + 1
        previous = current
        previous_lengths = current_lengths

    best_end = 1
    for reference_idx in range(2, reference_frames + 1):
        if previous[reference_idx] < previous[best_end]:
            best_end = reference_idx
    return previous[best_end] / max(1, previous_lengths[best_end])


def _shift_cqt_frequency(cqt, shift_bins):
    shifted = np.zeros_like(cqt)
    if shift_bins >= 0:
        shifted[shift_bins:] = cqt[:cqt.shape[0] - shift_bins]
    else:
        shifted[:shift_bins] = cqt[-shift_bins:]
    return shifted


def compute_cqt_dtw_distance(cqt_q, cqt_r, band_width=20, max_pitch_search=12):
    """Pitch-invariant subsequence DTW for QbH; ``band_width`` is retained for API compatibility."""
    del band_width
    if cqt_q.ndim != 2 or cqt_r.ndim != 2:
        raise ValueError("CQT tensors must have shape [frequency_bins, time_frames]")
    if cqt_q.shape[1] == 0 or cqt_r.shape[1] == 0:
        raise ValueError("CQT tensors must contain at least one frame")

    norm_q = cqt_q.astype(np.float32, copy=False)
    norm_r = cqt_r.astype(np.float32, copy=False)
    norm_q = norm_q / (np.linalg.norm(norm_q, axis=0, keepdims=True) + 1e-8)
    norm_r = norm_r / (np.linalg.norm(norm_r, axis=0, keepdims=True) + 1e-8)

    query_profile = norm_q.mean(axis=1)
    reference_profile = norm_r.mean(axis=1)
    reference_profile /= np.linalg.norm(reference_profile) + 1e-8
    best_shift = 0
    best_score = -np.inf
    for shift in range(-max_pitch_search, max_pitch_search + 1):
        shifted_profile = _shift_cqt_frequency(query_profile[:, None], shift).squeeze(1)
        shifted_profile /= np.linalg.norm(shifted_profile) + 1e-8
        score = float(np.dot(shifted_profile, reference_profile))
        if score > best_score:
            best_score = score
            best_shift = shift

    aligned_query = _shift_cqt_frequency(norm_q, best_shift)
    cost = np.clip(1.0 - aligned_query.T @ norm_r, 0.0, 2.0).astype(np.float32)
    return float(_subsequence_dtw_distance(cost))


### 8.2 Ham danh gia quy trinh truy van hai giai doan (evaluate_two_stage_dtw_retrieval)
Giai doan 1: Su dung Conformer True MaxMean loc nhanh Top-K ung vien tiem nang va do dac Candidate Recall@K de kiem tra tinh chinh xac cua tap ung vien. Giai doan 2: Su dung Band-Constrained DTW tren pho do dai thuc de tai xep hang chi tiet.

In [ ]:
from tqdm import tqdm
def evaluate_two_stage_dtw_retrieval(
    eval_model,
    gallery_songs,
    queries,
    top_candidates: int = 30,
    dtw_weight: float = 0.40,
    max_eval_queries: int = 150
):
    """
    Quy trình đánh giá hai giai đoạn (Two-Stage Retrieval Pipeline):
    - Giai đoạn 1: Conformer Global Embedding True MaxMean lọc nhanh Top-K ứng viên (mặc định K=30).
    - Đo lường Candidate Recall@K: Tỉ lệ bài hát mục tiêu lọt vào tập Top-K ứng viên trước khi chạy DTW.
    - Giai đoạn 2: Band-Constrained DTW (Sakoe-Chiba) với dải dò tone ±12 semitones tái xếp hạng chi tiết.
    """
    eval_model.eval()
    print(f"[*] Bắt đầu quy trình QbH Hai Giai Đoạn (Stage 1: Top-{top_candidates} Candidates -> Stage 2: Band-Constrained DTW Reranking)...")
    
    chunk_samples = prep.chunk_samples
    hop_samples = prep.hop_samples
    
    gallery_cqt_dict = {}
    gallery_emb_dict = {}
    
    with torch.no_grad():
        for s in tqdm(gallery_songs, desc="Caching Gallery CQT (Variable Length)", leave=False):
            vocal_p = os.path.join(VOCAL_DIR, f"{s}_vocal.wav") if VOCAL_DIR else ""
            mp3_p = os.path.join(MUSIC_DB_DIR, f"{s}.mp3") if MUSIC_DB_DIR else ""
            target_p = vocal_p if os.path.exists(vocal_p) else mp3_p
            if not os.path.exists(target_p):
                continue
                
            try:
                y_s, sr_s = sf.read(target_p, always_2d=False, dtype='float32')
                if len(y_s.shape) > 1: y_s = np.mean(y_s, axis=1)
                if sr_s != 16000: y_s = librosa.resample(y_s, orig_sr=sr_s, target_sr=16000)
            except Exception:
                try:
                    y_s, _ = librosa.load(target_p, sr=16000, duration=None)
                except Exception:
                    continue
                    
            y_s = y_s
            chunks = [y_s[st : st + chunk_samples] for st in range(0, len(y_s) - chunk_samples + 1, hop_samples)]
            if not chunks: chunks = [y_s[:chunk_samples]]
            
            # CQT độ dài động (variable-length) 30s đầu cho DTW
            cqt_full = cqt_ext.extract(y_s, variable_length=True).squeeze(0).cpu().numpy() # [96, T_r]
            gallery_cqt_dict[s] = cqt_full
            
            _, embs = encode_full_audio(eval_model,y_s)
            gallery_emb_dict[s] = embs

    hits_stage1 = {k: 0 for k in [1, 5, 10, 20]}
    hits_stage2 = {k: 0 for k in [1, 5, 10, 20]}
    mrrs_stage1 = []
    mrrs_stage2 = []
    candidate_recall_hits = 0
    
    test_subset = queries if (max_eval_queries is None or max_eval_queries <= 0) else queries[:max_eval_queries]
    for q_path, gt_song in tqdm(test_subset, desc="Evaluating Two-Stage Pipeline", leave=False):
        if gt_song not in gallery_emb_dict:
            continue
            
        try:
            y_q, sr_q = sf.read(q_path, always_2d=False, dtype='float32')
            if len(y_q.shape) > 1: y_q = np.mean(y_q, axis=1)
            if sr_q != 16000: y_q = librosa.resample(y_q, orig_sr=sr_q, target_sr=16000)
            if len(y_q) < chunk_samples:
                y_q = np.pad(y_q, (0, chunk_samples - len(y_q)), mode='constant')
            y_q = y_q[:16000 * 15]
            
            q_chunks = [y_q[st : st + chunk_samples] for st in range(0, len(y_q) - chunk_samples + 1, hop_samples)]
            if not q_chunks: q_chunks = [y_q[:chunk_samples]]
            
            with torch.no_grad():
                q_tensors = torch.stack([cqt_ext.extract(c, variable_length=False) for c in q_chunks], dim=0).to(device)
                q_embs = F.normalize(eval_model.extract_local_features(q_tensors), p=2, dim=-1)
                q_embs = q_embs.reshape(-1, q_embs.shape[-1]).cpu().numpy()
                
            # CQT độ dài động (variable-length) cho câu query
            cqt_q = cqt_ext.extract(y_q, variable_length=True).squeeze(0).cpu().numpy() # [96, T_q]
        except Exception:
            continue
            
        # ----------------- GIAI ĐOẠN 1: LỌC TOP-K ỨNG VIÊN BẰNG TRUE MAXMEAN -----------------
        s1_scores = {}
        for s, r_embs in gallery_emb_dict.items():
            sim_m = np.dot(q_embs, r_embs.T)
            s1_scores[s] = float(np.mean(np.max(sim_m, axis=1)))
            
        ranked_s1 = sorted(s1_scores.items(), key=lambda x: x[1], reverse=True)
        s1_songs = [x[0] for x in ranked_s1]
        
        r1 = s1_songs.index(gt_song) + 1 if gt_song in s1_songs else 999999
        mrrs_stage1.append(1.0 / r1)
        for k in [1, 5, 10, 20]:
            if r1 <= k: hits_stage1[k] += 1
            
        # Kiểm tra Candidate Recall@K
        top_k_candidates = s1_songs[:top_candidates]
        if gt_song in top_k_candidates:
            candidate_recall_hits += 1
            
        # ----------------- GIAI ĐOẠN 2: TÁI XẾP HẠNG BẰNG BAND-CONSTRAINED DTW -----------------
        s2_scores = {}
        for s, conf_score in ranked_s1[:top_candidates]:
            cqt_r = gallery_cqt_dict[s]
            dtw_dist = compute_cqt_dtw_distance(cqt_q, cqt_r, band_width=20, max_pitch_search=12)
            dtw_sim = max(0.0, 1.0 - dtw_dist)
            # Kết hợp điểm Conformer (Stage 1) và điểm DTW căn chỉnh giai điệu (Stage 2)
            s2_scores[s] = (1.0 - dtw_weight) * conf_score + dtw_weight * dtw_sim
            
        ranked_s2_top = sorted(s2_scores.items(), key=lambda x: x[1], reverse=True)
        s2_songs = [x[0] for x in ranked_s2_top] + s1_songs[top_candidates:]
        
        r2 = s2_songs.index(gt_song) + 1 if gt_song in s2_songs else 999999
        mrrs_stage2.append(1.0 / r2)
        for k in [1, 5, 10, 20]:
            if r2 <= k: hits_stage2[k] += 1
            
    n_eval = max(1, len(mrrs_stage1))
    return {
        'candidate_k': top_candidates,
        'candidate_recall': (candidate_recall_hits / n_eval) * 100.0,
        'stage1_top1': hits_stage1[1] / n_eval * 100.0,
        'stage1_top5': hits_stage1[5] / n_eval * 100.0,
        'stage1_top10': hits_stage1[10] / n_eval * 100.0,
        'stage1_top20': hits_stage1[20] / n_eval * 100.0,
        'stage1_mrr': float(np.mean(mrrs_stage1)) if mrrs_stage1 else 0.0,
        'stage2_top1': hits_stage2[1] / n_eval * 100.0,
        'stage2_top5': hits_stage2[5] / n_eval * 100.0,
        'stage2_top10': hits_stage2[10] / n_eval * 100.0,
        'stage2_top20': hits_stage2[20] / n_eval * 100.0,
        'stage2_mrr': float(np.mean(mrrs_stage2)) if mrrs_stage2 else 0.0,
        'total_evaluated': n_eval
    }

### 8.3 Thuc thi danh gia hai giai doan va in bang ket qua
Chay thu nghiem quy trinh hai giai doan: Stage 1 (Conformer True MaxMean) ket hop Stage 2 (Band-Constrained DTW Reranking) va in bang so sanh chi tiet kem Candidate Recall.

In [ ]:
res_dtw = evaluate_two_stage_dtw_retrieval(
    finetuned_eval_model,
    all_gallery_songs,
    all_test_queries,
    top_candidates=30,
    dtw_weight=0.40,
    max_eval_queries=150
)

delta_dtw_top1 = res_dtw['stage2_top1'] - res_dtw['stage1_top1']
delta_dtw_mrr = res_dtw['stage2_mrr'] - res_dtw['stage1_mrr']

print("\n" + "=" * 98)
print(f"{'QUY TRÌNH TRUY VẤN HAI GIAI ĐOẠN':^35} | {'Hit@1 (%)':^12} | {'Hit@5 (%)':^12} | {'Hit@10 (%)':^12} | {'Hit@20 (%)':^12} | {'MRR':^10}")
print("=" * 98)
print(f"{'Giai đoạn 1 (Conformer True MaxMean)':<35} | {res_dtw['stage1_top1']:^12.2f} | {res_dtw['stage1_top5']:^12.2f} | {res_dtw['stage1_top10']:^12.2f} | {res_dtw['stage1_top20']:^12.2f} | {res_dtw['stage1_mrr']:^10.4f}")
print(f"{'Giai đoạn 2 (+ Band-Constrained DTW)':<35} | {res_dtw['stage2_top1']:^12.2f} | {res_dtw['stage2_top5']:^12.2f} | {res_dtw['stage2_top10']:^12.2f} | {res_dtw['stage2_top20']:^12.2f} | {res_dtw['stage2_mrr']:^10.4f}")
print("-" * 98)
print(f"{'Mức Cải Thiện nhờ DTW (Δ)':<35} | {f'+{delta_dtw_top1:.2f}' if delta_dtw_top1>=0 else f'{delta_dtw_top1:.2f}':^12} | {'-':^12} | {'-':^12} | {'-':^12} | {f'+{delta_dtw_mrr:.4f}' if delta_dtw_mrr>=0 else f'{delta_dtw_mrr:.4f}':^10}")
print("=" * 98)
print(f"[+] Hiệu quả tạo ứng viên Giai đoạn 1: Stage 1 Candidate Recall@Top-{res_dtw['candidate_k']} = {res_dtw['candidate_recall']:.2f}%")
print(f"[+] Số câu queries đã đánh giá: {res_dtw['total_evaluated']:,} queries")

### 9.1 Ham trich xuat vector dac trung hai mien cho t-SNE (extract_tsne_data)
Trich xuat cac vector 128-D tu ca ban nhac goc (Studio) va cac doan hat vo (Humming) cua cac bai hat mau tu tap kiem thu de chuan bi du lieu cho truc quan hoa giam chieu t-SNE.

In [ ]:
from sklearn.manifold import TSNE
from matplotlib.lines import Line2D

test_unseen_candidates = sorted([d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))]) if (TEST_DIR and os.path.exists(TEST_DIR)) else []
target_songs = [s for s in test_unseen_candidates if (MUSIC_DB_DIR and os.path.exists(os.path.join(MUSIC_DB_DIR, f"{s}.mp3"))) or (VOCAL_DIR and os.path.exists(os.path.join(VOCAL_DIR, f"{s}_vocal.wav")))][:5]

def extract_tsne_data(eval_model, songs):
    embeddings = []
    labels = []
    types = []
    
    eval_model.eval()
    chunk_samples = prep.chunk_samples
    hop_samples = prep.hop_samples
    
    with torch.no_grad():
        for s in songs:
            vocal_p = os.path.join(VOCAL_DIR, f"{s}_vocal.wav") if VOCAL_DIR else ""
            mp3_p = os.path.join(MUSIC_DB_DIR, f"{s}.mp3") if MUSIC_DB_DIR else ""
            target_p = vocal_p if os.path.exists(vocal_p) else mp3_p
            if target_p and os.path.exists(target_p):
                try:
                    y, sr = sf.read(target_p, always_2d=False, dtype='float32')
                    if len(y.shape) > 1: y = np.mean(y, axis=1)
                    if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
                except Exception:
                    try:
                        y, _ = librosa.load(target_p, sr=16000, duration=30.0)
                    except Exception:
                        continue
                y = y[:16000 * 30]
                chunks = [y[st : st + chunk_samples] for st in range(0, len(y) - chunk_samples + 1, hop_samples)]
                if not chunks: chunks = [y[:chunk_samples]]
                q_tensors = torch.stack([cqt_ext.extract(c) for c in chunks[:3]], dim=0).to(device)
                embs = eval_model.extract_embedding(q_tensors).cpu().numpy()
                for e in embs:
                    embeddings.append(e)
                    labels.append(s)
                    types.append('Music Goc')
                    
            s_test_path = os.path.join(TEST_DIR, s) if TEST_DIR else ""
            if s_test_path and os.path.exists(s_test_path):
                for r, d, files in os.walk(s_test_path):
                    for f in files:
                        if f.endswith('.wav') and not f.startswith('reference'):
                            try:
                                y_h, sr_h = sf.read(os.path.join(r, f), always_2d=False, dtype='float32')
                                if len(y_h.shape) > 1: y_h = np.mean(y_h, axis=1)
                                if sr_h != 16000: y_h = librosa.resample(y_h, orig_sr=sr_h, target_sr=16000)
                            except Exception:
                                try:
                                    y_h, _ = librosa.load(os.path.join(r, f), sr=16000, duration=10.0)
                                except Exception:
                                    continue
                            c = cqt_ext.extract(y_h).unsqueeze(0).to(device)
                            e = eval_model.extract_embedding(c).squeeze(0).cpu().numpy()
                            embeddings.append(e)
                            labels.append(s)
                            types.append('Humming')
                            
    return np.array(embeddings), labels, types

### 9.2 Giam chieu t-SNE va truc quan hoa phan bo hai mien
Ve 2 bieu do khong gian 2D so sanh su phan bo dac trung giua nhac goc (Studio) va tieng hat vo (Humming): (A) Truoc Domain Adaptation (Zero-Shot) va (B) Sau Domain Adaptation (Fine-Tuned). Day la cong cu minh hoa dinh tinh phan anh su thu hep khoang cach bieu dien giua hai mien am thanh.

In [ ]:
if len(target_songs) >= 2:
    zs_model = zero_shot_model if 'zero_shot_model' in globals() else ConformerByteHumModel(ckpt_path=STAGE1_CKPT).to(device)
    ft_model = finetuned_eval_model
    
    X_zs, lab_zs, typ_zs = extract_tsne_data(zs_model, target_songs)
    X_ft, lab_ft, typ_ft = extract_tsne_data(ft_model, target_songs)
    
    if len(X_zs) > 2 and len(X_ft) > 2:
        perp_zs = max(2, min(15, len(X_zs) - 1))
        perp_ft = max(2, min(15, len(X_ft) - 1))
        
        # Tuong thich moi phien ban scikit-learn (>= 1.2 dung max_iter, < 1.2 dung n_iter)
        import inspect
        iter_key = 'max_iter' if 'max_iter' in inspect.signature(TSNE.__init__).parameters else 'n_iter'
        tsne_zs = TSNE(n_components=2, perplexity=perp_zs, random_state=42, **{iter_key: 1000})
        tsne_ft = TSNE(n_components=2, perplexity=perp_ft, random_state=42, **{iter_key: 1000})
        
        X_zs_2d = tsne_zs.fit_transform(X_zs)
        X_ft_2d = tsne_ft.fit_transform(X_ft)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), dpi=120)
        colors = plt.cm.tab10(np.linspace(0, 1, len(target_songs)))
        song_color = {s: colors[i] for i, s in enumerate(target_songs)}
        
        for ax, X_2d, lab, typ, title in zip(
            [ax1, ax2],
            [X_zs_2d, X_ft_2d],
            [lab_zs, lab_ft],
            [typ_zs, typ_ft],
            ['(A) Truoc Domain Adaptation (Zero-Shot)', '(B) Sau Domain Adaptation (Fine-Tuned)']
        ):
            for i in range(len(X_2d)):
                marker = 'o' if typ[i] == 'Music Goc' else '^'
                ax.scatter(X_2d[i, 0], X_2d[i, 1], color=song_color[lab[i]], marker=marker, s=80, alpha=0.8, edgecolors='white', linewidth=0.8)
            ax.set_title(title, fontsize=11, fontweight='bold')
            ax.grid(True, linestyle=':', alpha=0.5)
            
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', label='Music Goc (Studio)', markerfacecolor='gray', markersize=10),
            Line2D([0], [0], marker='^', color='w', label='Hat Vo (Humming)', markerfacecolor='gray', markersize=10)
        ]
        ax2.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9)
        
        plt.tight_layout()
        tsne_path = os.path.join(OUTPUT_DIR, "tsne_comparison.png")
        plt.savefig(tsne_path, dpi=150)
        plt.show()
        print(f"[+] Da luu bieu do t-SNE tai: {tsne_path}")
else:
    print("[!] Khong du so luong bai hat muc tieu trong tap test de ve bieu do t-SNE.")

### 10.1 Ham xay dung chi muc vector FAISS cho toan bo kho nhac (reindex_music_database)
Trich xuat vector dac trung 128-D cho toan bo 300 bai hat trong thu vien, chuan hoa vector L2 va dong goi vao chi muc FAISS IndexFlatIP de san sang phuc vu truy van vector toc do cao.

In [ ]:

from tqdm import tqdm
import hashlib
from pathlib import Path
import json


def reindex_music_database(model, songs_list=None, max_duration=None, dtw_duration=None, index_kind='flat'):
    """Build FAISS global index and cache local features/CQT needed by online reranking."""
    model.eval()
    if songs_list is None:
        songs_list = all_gallery_songs
    print(f"[*] Bat dau Re-Indexing cho {len(songs_list)} bai hat (duration limit: {max_duration})...")

    chunk_samples = prep.chunk_samples
    hop_samples = prep.hop_samples
    all_embs, chunk_to_song = [], []
    song_metadata, song_embeddings = {}, {}
    song_local_embeddings, song_dtw_cqts = {}, {}
    failed_songs = []
    model_digest = hashlib.sha256()
    for key,value in sorted(model.state_dict().items()):
        model_digest.update(key.encode())
        model_digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    cache_root = Path(OUTPUT_DIR,'full_song_cache',model_digest.hexdigest()[:20])
    cache_root.mkdir(parents=True,exist_ok=True)

    with torch.no_grad():
        for s in tqdm(songs_list, desc="Re-indexing 300 Songs", leave=False):
            vocal_p = os.path.join(VOCAL_DIR, f"{s}_vocal.wav") if VOCAL_DIR else ""
            mp3_p = os.path.join(MUSIC_DB_DIR, f"{s}.mp3") if MUSIC_DB_DIR else ""
            target_p = vocal_p if os.path.exists(vocal_p) else mp3_p
            if not target_p or not os.path.exists(target_p):
                continue
            try:
                stat = os.stat(target_p)
                identity = dict(path=os.path.realpath(target_p), size=stat.st_size, mtime=stat.st_mtime_ns,
                                sample_rate=prep.sample_rate,chunk=prep.chunk_samples,hop=prep.hop_samples,
                                frames=cqt_ext.target_frames, mean_size=cqt_ext.mean_size,
                                max_duration=max_duration,dtw_duration=dtw_duration,version=2)
                key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
                cache_file=cache_root/(key+'.npz')
                if cache_file.exists():
                    with np.load(cache_file,allow_pickle=False) as cache:
                        global_embs, local_embs, dtw_cqt = cache['global'],cache['local'],cache['cqt']
                else:
                    y = prep.load_audio(target_p)
                    if max_duration is not None:
                        y = y[:int(prep.sample_rate * max_duration)]
                    global_embs, local_embs = encode_full_audio(model,y)
                    dtw_audio=y if dtw_duration is None else y[:int(prep.sample_rate*dtw_duration)]
                    dtw_cqt=cqt_ext.extract(dtw_audio,variable_length=True).squeeze(0).cpu().numpy()
                    temporary=cache_file.with_suffix('.tmp.npz')
                    np.savez_compressed(temporary,**{'global':global_embs,'local':local_embs,'cqt':dtw_cqt})
                    os.replace(temporary,cache_file)
                s_idx = len(song_metadata)
                song_metadata[s_idx] = {'title':s,'num_chunks':len(global_embs),'cache_file':str(cache_file)}
                song_embeddings[s_idx] = global_embs.astype(np.float32,copy=False)
                song_local_embeddings[s_idx] = local_embs.astype(np.float32,copy=False)
                song_dtw_cqts[s_idx] = dtw_cqt
                for embedding in global_embs:
                    all_embs.append(embedding)
                    chunk_to_song.append(s_idx)
            except Exception as exc:
                failed_songs.append((s, repr(exc)))

    if failed_songs:
        raise RuntimeError(f"Re-index that bai voi {len(failed_songs)} bai, vi du: {failed_songs[:3]}")
    if not all_embs:
        raise RuntimeError("Khong co embedding nao de xay FAISS index")

    faiss_matrix = np.asarray(all_embs, dtype=np.float32)
    faiss_index = build_retrieval_index(faiss_matrix,index_kind)
    index_file = os.path.join(OUTPUT_DIR, "conformer_db_300_songs.index")
    map_file = os.path.join(OUTPUT_DIR, "conformer_song_map.pkl")
    faiss.write_index(faiss_index, index_file)
    with open(map_file, "wb") as handle:
        pickle.dump({'chunk_to_song': chunk_to_song, 'song_metadata': song_metadata,
                     'song_embeddings': song_embeddings, 'song_local_embeddings': song_local_embeddings,
                     'song_dtw_cqts': song_dtw_cqts,
                     'index_metadata': {'sample_rate': prep.sample_rate, 'chunk_duration_sec': prep.chunk_duration,
                                        'hop_duration_sec': prep.hop_duration, 'max_duration_sec': max_duration,
                                        'dtw_duration_sec': dtw_duration, 'embedding_dim': int(faiss_index.d),
                                        'index_kind': index_kind, 'checkpoint_path': finetuned_ckpt_path}}, handle)
    print(f"[+] Re-Indexing hoan tat: {len(song_metadata)} bai ({len(faiss_matrix):,} chunk vectors).")
    return faiss_index, chunk_to_song, song_metadata, song_embeddings, song_local_embeddings, song_dtw_cqts


@torch.no_grad()
def evaluate_faiss_candidate_recall(eval_model, dataset, faiss_index, chunk_to_song, song_metadata, candidate_k=50):
    """Measure whether the deployed FAISS gate retains the ground-truth song."""
    eval_model.eval()
    hits = 0
    for song, _, query_path in tqdm(dataset.humming_samples, desc='FAISS candidate recall', leave=False):
        query_cqts = torch.stack(dataset._get_raw_cqts(query_path), dim=0).to(device)
        query_global = F.normalize(eval_model.extract_embedding(query_cqts), p=2, dim=-1).cpu().numpy()
        _, indices = faiss_index.search(query_global, min(candidate_k, faiss_index.ntotal))
        candidates = {song_metadata[chunk_to_song[index]]['title'] for row in indices for index in row if index != -1}
        hits += int(song in candidates)
    total = len(dataset.humming_samples)
    if total == 0:
        raise RuntimeError('FAISS candidate recall khong co query hop le')
    return {'candidate_k': candidate_k, 'hits': hits, 'total': total, 'recall': hits / total}


### 10.2 Thuc thi chi muc hoa kho nhac 300 bai va luu file chi muc
Chay reindex_music_database tren mo hinh da tinh chinh va luu cac file `conformer_db_300_songs.index` cung `conformer_song_map.pkl` vao thu muc output.

In [ ]:
db_index, db_map, db_meta, db_song_embeddings, db_song_local_embeddings, db_song_dtw_cqts = reindex_music_database(finetuned_eval_model, max_duration=None)
faiss_candidate_metrics = evaluate_faiss_candidate_recall(
    finetuned_eval_model, test_dataset, db_index, db_map, db_meta, candidate_k=50
)
print(f"[+] FAISS Candidate Recall@{faiss_candidate_metrics['candidate_k']}: {faiss_candidate_metrics['recall']:.2%} ({faiss_candidate_metrics['hits']}/{faiss_candidate_metrics['total']})")


### 10.3 Ham tim kiem bai hat tu file am thanh voi phan ra do tre chi tiet (search_humming)
Nhan mot file am thanh humming bat ky, cat cua so truot, trich xuat dac trung, truy van nhanh qua FAISS IndexFlatIP, tinh diem True MaxMean va tai xep hang chi tiet bang Band-Constrained DTW (tuy chon). Ham phan ra chi tiet do tre tung cong doan (audio loading, CQT, Conformer inference, FAISS search, MaxMean aggregation, DTW reranking).

In [32]:
def search_humming(audio_path, top_k=5, faiss_k=50, enable_dtw=True, dtw_candidates=10, dtw_weight=0.40):
    """Search with FAISS candidates, local MaxMean scoring, and fully cached DTW references."""
    t_total_start = time.perf_counter()
    if not os.path.exists(audio_path):
        return {'status': 'error', 'message': f'File khong ton tai: {audio_path}'}
    if db_index.ntotal == 0:
        return {'status': 'error', 'message': 'FAISS index rong; hay re-index gallery truoc.'}

    t0 = time.perf_counter()
    try:
        y_q = prep.load_audio(audio_path)
    except Exception as exc:
        return {'status': 'error', 'message': f'Khong the doc audio: {exc}'}
    q_chunks = prep.slice_into_chunks(y_q)
    t_audio = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    q_tensors = torch.stack([cqt_ext.extract(chunk) for chunk in q_chunks], dim=0).to(device)
    cqt_q_var = cqt_ext.extract(y_q, variable_length=True).squeeze(0).cpu().numpy() if enable_dtw else None
    t_cqt = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    finetuned_eval_model.eval()
    with torch.no_grad():
        q_global = F.normalize(finetuned_eval_model.extract_embedding(q_tensors), p=2, dim=-1).cpu().numpy()
        q_local = F.normalize(finetuned_eval_model.extract_local_features(q_tensors), p=2, dim=-1)
        q_local = q_local.reshape(-1, q_local.shape[-1]).cpu().numpy()
    t_model = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    _, indices = db_index.search(q_global, min(faiss_k, db_index.ntotal))
    candidate_song_ids = {db_map[index] for row in indices for index in row if index != -1}
    t_faiss = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    song_scores = {}
    for song_idx in candidate_song_ids:
        reference_local = db_song_local_embeddings[song_idx]
        similarity = q_local @ reference_local.T
        song_scores[song_idx] = float(similarity.max(axis=1).mean())
    ranked_ids = sorted(song_scores, key=song_scores.get, reverse=True)
    t_maxmean = (time.perf_counter() - t0) * 1000

    t_dtw = 0.0
    if enable_dtw and dtw_candidates > 0 and ranked_ids:
        t0 = time.perf_counter()
        reranked_scores = {}
        for song_idx in ranked_ids[:dtw_candidates]:
            distance = compute_cqt_dtw_distance(cqt_q_var, db_song_dtw_cqts[song_idx], max_pitch_search=12)
            dtw_similarity = max(0.0, 1.0 - distance)
            reranked_scores[song_idx] = (1.0 - dtw_weight) * song_scores[song_idx] + dtw_weight * dtw_similarity
        ranked_ids = sorted(reranked_scores, key=reranked_scores.get, reverse=True) + ranked_ids[dtw_candidates:]
        for song_idx, score in reranked_scores.items():
            song_scores[song_idx] = score
        t_dtw = (time.perf_counter() - t0) * 1000

    t_total = (time.perf_counter() - t_total_start) * 1000
    return {'status': 'success', 'query_path': audio_path,
            'latency': {'audio_preprocess_ms': t_audio, 'cqt_extraction_ms': t_cqt,
                        'conformer_inference_ms': t_model, 'faiss_search_ms': t_faiss,
                        'maxmean_scoring_ms': t_maxmean, 'dtw_reranking_ms': t_dtw, 'total_ms': t_total},
            'results': [{'rank': rank, 'title': db_meta[song_idx]['title'], 'score': song_scores[song_idx]}
                        for rank, song_idx in enumerate(ranked_ids[:top_k], start=1)]}


In [33]:
import hashlib
import json

def retrieval_signature(settings):
    digest = hashlib.sha256()
    with open(finetuned_ckpt_path,'rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    digest.update(faiss.serialize_index(db_index).tobytes())
    digest.update(json.dumps(db_meta,sort_keys=True).encode())
    digest.update(json.dumps(settings,sort_keys=True).encode())
    return digest.hexdigest()

def fit_open_set_threshold(known_scores, unknown_scores, target_far=0.01):
    known, unknown = np.asarray(known_scores), np.asarray(unknown_scores)
    if not len(known) or not len(unknown) or not 0 <= target_far < 1:
        raise ValueError('Calibration needs held-out known and unknown queries')
    if not np.isfinite(known).all() or not np.isfinite(unknown).all():
        raise ValueError('Nonfinite calibration scores')
    thresholds = np.unique(np.r_[known,unknown,np.nextafter(max(known.max(),unknown.max()),np.inf)])
    feasible = [float(t) for t in thresholds if np.mean(unknown >= t) <= target_far]
    threshold = min(feasible)
    return dict(threshold=threshold, target_far=target_far,
                empirical_far=float(np.mean(unknown>=threshold)),
                known_acceptance=float(np.mean(known>=threshold)),
                known_count=len(known), unknown_count=len(unknown))

def calibrate_open_set(records, settings=None, target_far=0.01):
    # records: path, song_id (null for unknown), split='calibration'.
    settings = settings or dict(faiss_k=50,enable_dtw=True,dtw_candidates=10,dtw_weight=0.4)
    known, unknown, paths = [], [], set()
    gallery = {m['title'] for m in db_meta.values()}
    test_paths = {os.path.realpath(q) for q,_ in all_test_queries}
    for row in records:
        path = os.path.realpath(row['path'])
        if row.get('split') != 'calibration' or path in test_paths or path in paths:
            raise ValueError('Calibration/test leakage, duplicate or missing split')
        paths.add(path)
        label = row.get('song_id')
        if label is not None and label not in gallery:
            raise ValueError('Known calibration song absent from gallery')
        result = search_humming_closed(path,top_k=1,**settings)
        if result['status'] != 'success' or not result['results']:
            raise RuntimeError(f'Calibration query failed: {path}')
        (unknown if label is None else known).append(result['results'][0]['score'])
    fitted = fit_open_set_threshold(known,unknown,target_far)
    fitted.update(settings=settings,signature=retrieval_signature(settings),paths=sorted(paths))
    with open(os.path.join(OUTPUT_DIR,'open_set_calibration.json'),'w',encoding='utf-8') as f:
        json.dump(fitted,f,indent=2)
    return fitted

search_humming_closed = search_humming
OPEN_SET_CALIBRATION = None

def search_humming(audio_path, top_k=5, faiss_k=50, enable_dtw=True, dtw_candidates=10, dtw_weight=0.4, calibration=None):
    settings = dict(faiss_k=faiss_k,enable_dtw=enable_dtw,dtw_candidates=dtw_candidates,dtw_weight=dtw_weight)
    result = search_humming_closed(audio_path,top_k=top_k,**settings)
    calibration = OPEN_SET_CALIBRATION if calibration is None else calibration
    result['open_set_calibrated'] = calibration is not None
    if calibration is not None:
        if settings != calibration['settings'] or retrieval_signature(settings) != calibration['signature']:
            raise ValueError('Recalibrate after changing checkpoint, gallery, index or scoring settings')
        if result['status']=='success' and (not result['results'] or result['results'][0]['score'] < calibration['threshold']):
            result['status'] = 'not_found'
            result['results'] = []
    return result

# Set to a JSON list of independent validation known/unknown queries to fit the threshold.
CALIBRATION_MANIFEST = os.environ.get('QBH_CALIBRATION_MANIFEST')
if CALIBRATION_MANIFEST:
    with open(CALIBRATION_MANIFEST,encoding='utf-8') as f:
        OPEN_SET_CALIBRATION = calibrate_open_set(json.load(f))


### 10.4 Chay thu nghiem tim kiem mot cau humming thuc te va in bang phan ra do tre
Chay thu nghiem ham search_humming tren 1 doan humming mau tu tap test va in danh sach Top-5 bai hat duoc du doan kem theo bang phan ra do tre tung cong doan (Audio Load, CQT, Conformer, FAISS, MaxMean, DTW).

In [34]:
if all_test_queries:
    demo_query, demo_gt = all_test_queries[0]
    res_search = search_humming(demo_query, top_k=5, faiss_k=50, enable_dtw=True, dtw_candidates=10)
    lat = res_search['latency']
    
    print("\n" + "=" * 62)
    print(f"{'PHAN RA DO TRE TRUY VAN (LATENCY BREAKDOWN BENCHMARK)':^62}")
    print("=" * 62)
    print(f"Audio Preprocessing       : {lat['audio_preprocess_ms']:>8.2f} ms")
    print(f"CQT Feature Extraction    : {lat['cqt_extraction_ms']:>8.2f} ms")
    print(f"Conformer Model Inference : {lat['conformer_inference_ms']:>8.2f} ms")
    print(f"FAISS IndexFlatIP Search  : {lat['faiss_search_ms']:>8.2f} ms")
    print(f"True MaxMean Scoring      : {lat['maxmean_scoring_ms']:>8.2f} ms")
    print(f"Band-Constrained DTW      : {lat['dtw_reranking_ms']:>8.2f} ms")
    print("-" * 62)
    print(f"Tong Do Tre (Total Latency): {lat['total_ms']:>8.2f} ms")
    print("=" * 62)
    
    print(f"\n[*] KET QUA TIM KIEM DEMO:")
    print(f"    - Query Audio : {os.path.basename(demo_query)}")
    print(f"    - Bai Hat That: {demo_gt}")
    print(f"    - Ket qua Top-5:")
    for r in res_search['results']:
        matched = ' [CHINH XAC]' if r['title'] == demo_gt else ''
        print(f"       Top {r['rank']}: {r['title']} (Diem: {r['score']:.4f}){matched}")

NameError: name 'db_index' is not defined

### 10.5 Bang tong hop ket qua thuc nghiem toan dien (Benchmark Summary)
Xuat bao cao tong hop toan bo cac moc thuc nghiem tu Baseline Zero-Shot, Fine-Tuned (Conformer-ByteHum) den Two-Stage (+ Band-Constrained DTW), su dung hoan toan cac bien ket qua thuc te (khong hardcode so lieu gia lap, hien thi 'N/A' neu thi nghiem chua chay).

In [ ]:
from IPython.display import display, Markdown

def fmt_pct(val):
    return f"{val:.2f}%" if isinstance(val, (int, float)) else "N/A"

def fmt_mrr(val):
    return f"{val:.4f}" if isinstance(val, (int, float)) else "N/A"

# 1. Baseline: Zero-Shot
top1_zs  = fmt_pct(res_zs.get('top1')) if 'res_zs' in globals() else "N/A"
top5_zs  = fmt_pct(res_zs.get('top5')) if 'res_zs' in globals() else "N/A"
top10_zs = fmt_pct(res_zs.get('top10')) if 'res_zs' in globals() else "N/A"
top20_zs = fmt_pct(res_zs.get('top20')) if 'res_zs' in globals() else "N/A"
mrr_zs   = fmt_mrr(res_zs.get('mrr')) if 'res_zs' in globals() else "N/A"

# 2. Fine-Tuned: Conformer-ByteHum
top1_ft  = fmt_pct(res_ft.get('top1')) if 'res_ft' in globals() else "N/A"
top5_ft  = fmt_pct(res_ft.get('top5')) if 'res_ft' in globals() else "N/A"
top10_ft = fmt_pct(res_ft.get('top10')) if 'res_ft' in globals() else "N/A"
top20_ft = fmt_pct(res_ft.get('top20')) if 'res_ft' in globals() else "N/A"
mrr_ft   = fmt_mrr(res_ft.get('mrr')) if 'res_ft' in globals() else "N/A"

# 3. Two-Stage: Band-Constrained DTW
top1_dtw  = fmt_pct(res_dtw.get('stage2_top1')) if 'res_dtw' in globals() else "N/A"
top5_dtw  = fmt_pct(res_dtw.get('stage2_top5')) if 'res_dtw' in globals() else "N/A"
top10_dtw = fmt_pct(res_dtw.get('stage2_top10')) if 'res_dtw' in globals() else "N/A"
top20_dtw = fmt_pct(res_dtw.get('stage2_top20')) if 'res_dtw' in globals() else "N/A"
mrr_dtw   = fmt_mrr(res_dtw.get('stage2_mrr')) if 'res_dtw' in globals() else "N/A"

summary_md = f"""
### BANG TONG HOP KET QUA THUC NGHIEM TOAN DIEN (QBH BENCHMARK)

| Cau Hinh Thuc Nghiem | Hit@1 (%) | Hit@5 (%) | Hit@10 (%) | Hit@20 (%) | MRR | Ghi Chu Ky Thuat |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **(1) Baseline: Zero-Shot Conformer** | {top1_zs} | {top5_zs} | {top10_zs} | {top20_zs} | {mrr_zs} | Trong so goc SHS100K (CSI Stage 1) |
| **(2) Fine-Tuned: Conformer-ByteHum** | **{top1_ft}** | **{top5_ft}** | **{top10_ft}** | **{top20_ft}** | **{mrr_ft}** | Multi-Task (Triplet + Domain GRL + LAC True MaxMean) |
| **(3) Two-Stage: + Band-Constrained DTW** | **{top1_dtw}** | **{top5_dtw}** | **{top10_dtw}** | **{top20_dtw}** | **{mrr_dtw}** | Stage 1 MaxMean Top-30 -> Stage 2 DTW (+-12 semitones) |

---
### DANH SACH CAC TEP TIN XUAT RA TAI THU MUC OUTPUT:
- **Trong so mo hinh toi uu** : `{finetuned_ckpt_path if 'finetuned_ckpt_path' in globals() else 'best_conformer_bytehum_mrr.pth'}`
- **Chi muc FAISS kho nhac** : `{os.path.join(OUTPUT_DIR, 'conformer_db_300_songs.index')}`
- **Bang anh xa bai hat**   : `{os.path.join(OUTPUT_DIR, 'conformer_song_map.pkl')}`
- **Bieu do huan luyen**    : `{os.path.join(OUTPUT_DIR, 'loss_curves.png')}`
- **Bieu do duong cong CMC**: `{os.path.join(OUTPUT_DIR, 'cmc_curve.png')}`
- **Bieu do phan bo t-SNE** : `{os.path.join(OUTPUT_DIR, 'tsne_comparison.png')}`
"""
display(Markdown(summary_md))

### Public benchmarks, calibration and ablations
MIR-QBSH MIDI synthesis requires pretty_midi. MTG-QBH requires original recordings mapped using official song/class IDs. Calibration queries must be independent of test queries.

In [ ]:
from pathlib import Path

def load_benchmark_manifest(path):
    """Audio manifest: dataset, protocol, gallery[{song_id,path}], queries[{song_id,path}]."""
    path = Path(path)
    with path.open(encoding='utf-8') as f:
        manifest = json.load(f)
    if manifest['dataset'] not in ('MIR-QBSH','IOACAS','MTG-QBH'):
        raise ValueError('Unknown benchmark')
    if not manifest.get('protocol'):
        raise ValueError('Declare reference protocol: original-audio or synthesized-MIDI')
    for group in ('gallery','queries'):
        if not manifest[group]:
            raise ValueError(f'Empty {group}')
        for row in manifest[group]:
            audio = Path(row['path'])
            row['path'] = str(audio if audio.is_absolute() else (path.parent/audio).resolve())
            if not Path(row['path']).is_file():
                raise FileNotFoundError(row['path'])
    ids = {row['song_id'] for row in manifest['gallery']}
    if any(row['song_id'] not in ids for row in manifest['queries']):
        raise ValueError('Query ground truth absent from gallery')
    if {row['path'] for row in manifest['gallery']} & {row['path'] for row in manifest['queries']}:
        raise ValueError('Query/reference audio overlap')
    return manifest

def prepare_mir_qbsh(extracted_root, output_dir):
    """Explicit sine-synthesized MIDI protocol, NOT original-recording ByteHum protocol."""
    import pretty_midi
    root, out = Path(extracted_root), Path(output_dir)
    out.mkdir(parents=True,exist_ok=True)
    gallery=[]
    for midi in sorted((root/'midiFile').glob('*.mid')):
        target=out/(midi.stem+'.wav')
        sf.write(str(target),pretty_midi.PrettyMIDI(str(midi)).synthesize(fs=SAMPLE_RATE),SAMPLE_RATE)
        gallery.append(dict(song_id=midi.stem,path=str(target.resolve())))
    queries=[dict(song_id=q.stem,path=str(q.resolve()),singer_id='/'.join(q.parts[-3:-1]))
             for q in sorted((root/'waveFile').rglob('*.wav'))]
    manifest=dict(dataset='MIR-QBSH',protocol='sine-synthesized-MIDI',gallery=gallery,queries=queries)
    target=out/'manifest.json'
    target.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    return str(target)

def evaluate_benchmark_manifest(manifest_path, eval_model, index_kind='flat', enable_dtw=False, candidate_k=50, dtw_candidates=10, dtw_weight=0.4):
    manifest=load_benchmark_manifest(manifest_path)
    local_cache, cqt_cache, vectors, vector_ids = {}, {}, [], []
    for row in tqdm(manifest['gallery'],desc='Public benchmark gallery'):
        y=prep.load_audio(row['path'])
        g,l=encode_full_audio(eval_model,y)
        sid=row['song_id']
        local_cache.setdefault(sid,[]).append(l)
        if enable_dtw:
            cqt_cache.setdefault(sid,[]).append(cqt_ext.extract(y,variable_length=True).squeeze(0).numpy())
        vectors.extend(g); vector_ids.extend([sid]*len(g))
    local_cache={sid:np.concatenate(parts) for sid,parts in local_cache.items()}
    index=build_retrieval_index(np.asarray(vectors),index_kind)
    ranks, recall, latency = [], [], []
    for row in tqdm(manifest['queries'],desc='Public benchmark queries'):
        start=time.perf_counter()
        y=prep.load_audio(row['path'])
        qg,ql=encode_full_audio(eval_model,y)
        _,nearest=index.search(qg,min(candidate_k,index.ntotal))
        candidates={vector_ids[i] for i in nearest.ravel() if i>=0}
        recall.append(row['song_id'] in candidates)
        scores={sid:float((ql@local_cache[sid].T).max(1).mean()) for sid in candidates}
        ordered=sorted(scores,key=scores.get,reverse=True)
        if enable_dtw:
            qc=cqt_ext.extract(y,variable_length=True).squeeze(0).numpy()
            top=ordered[:dtw_candidates]
            fused={sid:(1-dtw_weight)*scores[sid]+dtw_weight*max(0.,1-min(compute_cqt_dtw_distance(qc,r) for r in cqt_cache[sid])) for sid in top}
            ordered=sorted(top,key=fused.get,reverse=True)+ordered[dtw_candidates:]
        ranks.append(ordered.index(row['song_id'])+1 if row['song_id'] in ordered else np.inf)
        latency.append((time.perf_counter()-start)*1000)
    ranks=np.asarray(ranks)
    return dict(dataset=manifest['dataset'],protocol=manifest['protocol'],queries=len(ranks),gallery=len(local_cache),
                top1=float(np.mean(ranks<=1)),top5=float(np.mean(ranks<=5)),top10=float(np.mean(ranks<=10)),
                mrr=float(np.mean(1/ranks)),candidate_recall=float(np.mean(recall)),
                latency_p50_ms=float(np.median(latency)),latency_p95_ms=float(np.percentile(latency,95)),
                index_kind=index_kind,dtw=enable_dtw)

def run_ablation_suite(checkpoints, manifest_path, output_path):
    """Each training ablation requires its own independently trained checkpoint."""
    results=[]
    for name,path in checkpoints.items():
        ckpt=torch.load(path,map_location='cpu',weights_only=False)
        if 'run_config' not in ckpt or ckpt.get('run_name') != name:
            raise ValueError('Missing/mismatched ablation provenance')
        state=ckpt['model_state_dict']
        classes,_,dim=state['lac.prototypes'].shape
        m=ConformerByteHumModel(num_classes=classes,embedding_dim=dim,allow_random_init=True).to(device)
        m.load_state_dict(state,strict=True)
        for dtw in (False,True):
            result=evaluate_benchmark_manifest(manifest_path,m,enable_dtw=dtw)
            results.append(dict(result,run=name,checkpoint=str(path),config=ckpt['run_config']))
            Path(output_path).write_text(json.dumps(results,indent=2),encoding='utf-8')
        del m
    return results

def compare_index_backends(gallery_vectors, query_vectors, k=50, nlist=64):
    """Measure ANN recall against exact neighbors and search time on identical vectors."""
    q=np.ascontiguousarray(query_vectors,dtype=np.float32)
    faiss.normalize_L2(q)
    exact=build_retrieval_index(gallery_vectors.copy(),'flat')
    k=min(k,exact.ntotal)
    _,truth=exact.search(q,k)
    results=[]
    for kind in ('flat','hnsw','ivf'):
        if kind=='ivf' and len(gallery_vectors)<39*nlist:
            results.append(dict(kind=kind,status='insufficient_training_vectors')); continue
        start=time.perf_counter()
        index=build_retrieval_index(gallery_vectors.copy(),kind,nlist=nlist)
        build_sec=time.perf_counter()-start
        start=time.perf_counter(); _,found=index.search(q,k)
        elapsed=time.perf_counter()-start
        recall=np.mean([len(set(a)&set(b))/k for a,b in zip(truth,found)])
        results.append(dict(kind=kind,dimension=q.shape[1],recall_at_k=float(recall),build_seconds=build_sec,ms_per_query=elapsed*1000/len(q)))
    return results

PUBLIC_BENCHMARK_MANIFEST = os.environ.get('QBH_BENCHMARK_MANIFEST')
if PUBLIC_BENCHMARK_MANIFEST:
    public_result=evaluate_benchmark_manifest(PUBLIC_BENCHMARK_MANIFEST,finetuned_eval_model)
    Path(OUTPUT_DIR,'public_benchmark.json').write_text(json.dumps(public_result,indent=2),encoding='utf-8')
    print(public_result)
else:
    print('Public benchmark pending: set QBH_BENCHMARK_MANIFEST. No benchmark result has been measured.')
